# 28 Referensi Supervised ILS


In [1]:
from pathlib import Path
import os, types
from IPython.display import display

PROJECT_ROOT = '/content/drive/MyDrive/REVISED'
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', force_remount=False)
ROOT_28 = Path(os.environ.get('SUPERVISED28_ROOT', PROJECT_ROOT)).expanduser().resolve()
if not (ROOT_28 / 'LIB').is_dir():
    raise FileNotFoundError('Folder LIB proyek belum tersedia: ' + str(ROOT_28))
print('Proyek:', ROOT_28)


Mounted at /content/drive
Proyek: /content/drive/MyDrive/REVISED


## Desain perbandingan dan anggaran label

| Kelompok | Target latihan | ILS train sebagai target/model |
|---|---|---|
| CDS-NB, CDS-DT, CDS-SVM | Base LS dari mapping CARE-KM train tahap 26 | 0 peserta |
| ILS-NB, ILS-DT, ILS-SVM | Base LS dari ILS train aktual | 322 peserta / 1.288 arah dimensi |
| CARE-KM direct dan behavioral rules | Profil perilaku tersimpan tahap 27 | 0 peserta |
| ILS train-majority | Kutub terbanyak pada ILS train | 322 peserta |

Nol label ILS untuk target latihan berbeda dari tanpa akses ILS dalam
seluruh desain penelitian: split bersama sebelumnya memakai ILS untuk
stratifikasi. Label test tidak masuk fitting, pemilihan parameter, atau
penentuan mayoritas. Prediktor dibatasi pada 18 indikator; identitas,
cluster, split, kode/skor ILS, dan profil tidak menjadi fitur.

Classifier mempelajari Base LS komposit, lalu hasilnya diurai menjadi empat arah. Target CDS dan ILS boleh memiliki jumlah
kelas teramati berbeda; kelas ILS yang jarang tidak dibuang. Tidak ada
hyperparameter search atau pemilihan hasil terbaik dari test.


In [2]:
KONFIGURASI_28 = {
    'force': False,
}


## Model, pemakaian ulang, dan ketidakpastian

Factory classifier diambil dari notebook 22 dan diperiksa berdasarkan
definisi fungsi. NB memakai alpha=1 dan min_categories=4; DT memakai input
integer-coded, class_weight balanced, random_state 161600; SVM memakai
one-hot train-only, RBF, C=1, gamma=scale, class_weight balanced.
Seed NumPy 161600 dipakai sementara saat fit lalu state sesi dipulihkan.

Model dan prediksi memiliki cache terpisah yang memeriksa isi. Jalankan
notebook yang sama kembali jika Colab terputus. Perubahan output tampilan
notebook tidak membatalkan cache. `force=True` menghitung ulang model dan
evaluasi tahap 28, tanpa mengulang tahap 25–27.

Ukuran utama adalah mean balanced accuracy empat dimensi, disertai
accuracy, macro-F1 biner per dimensi, confusion matrices, coverage, dan CI.
Gunakan 10.000 bobot bootstrap peserta yang disimpan pada tahap 27; empat
dimensi serta prediksi semua jalur tetap berpasangan. Tidak ada draws baru.

Tiga kontras utama membandingkan sumber label dalam keluarga sama:
ILS-NB vs CDS-NB, ILS-DT vs CDS-DT, dan ILS-SVM vs CDS-SVM. Interval
individual bersifat deskriptif; mean BA utama juga memiliki interval
percentile Bonferroni keluarga tiga kontras. Perbandingan dengan profil
langsung dilaporkan terpisah dengan anggaran label berbeda.

CI bersyarat pada split dan konfigurasi tetap. Ini holdout Pendikom yang
sama, bukan cohort baru. Supervised ILS merupakan referensi berlabel,
bukan batas atas performa yang dijamin.


In [3]:
HELPER26_SOURCE = '"""Fixed-rule semantic evaluation from the immutable partition bank of stage 25."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport ast\nimport gzip\nimport hashlib\nimport importlib.metadata\nimport io\nimport itertools\nimport json\nimport os\nimport platform\nimport re\nimport sys\nimport uuid\nimport numpy as np\nimport pandas as pd\n\nSTAGE = \'26_Pengaruh_Initializer_Profil_Semantik\'\nUPSTREAM = \'25_Perbandingan_Initializer_Terintegrasi\'\nUP = \'OUTPUT/TAMBAHAN/\' + UPSTREAM\nUP_CONTROL = \'OUTPUT/TAMBAHAN/KONTROL/\' + UPSTREAM\nMETHODS = [\'Random\',\'Huang\',\'Cao\',\'Bai\',\'K-Modes++\',\'HR-2023\',\'CARE-KM\',\'BK-Modes\']\nSTOCHASTIC = {\'Random\',\'Huang\',\'K-Modes++\',\'HR-2023\'}\nORDER = [\'CARE-KM\',\'Cao\',\'Bai\',\'BK-Modes\',\'K-Modes++\',\'Random\',\'Huang\',\'HR-2023\']\nKEYS = [\'active_reflective\',\'sensing_intuitive\',\'visual_verbal\',\'sequential_global\']\nDIMS = [\'Active/Reflective\',\'Sensing/Intuitive\',\'Visual/Verbal\',\'Sequential/Global\']\nPOLES = [(\'Active\',\'Reflective\'),(\'Sensing\',\'Intuitive\'),(\'Visual\',\'Verbal\'),(\'Sequential\',\'Global\')]\nABBR = {\'Active\':\'Act.\',\'Reflective\':\'Ref.\',\'Sensing\':\'Sens.\',\'Intuitive\':\'Int.\',\n        \'Visual\':\'Vis.\',\'Verbal\':\'Ver.\',\'Sequential\':\'Seq.\',\'Global\':\'Gl.\'}\nLS_LOOKUP = {combo:\'LS%02d\' % (i+1) for i,combo in enumerate(itertools.product(*POLES))}\nMAIN = [\'content_visit\',\'content_stay\',\'selfass_visit_proxy\',\'selfass_stay_proxy\',\'exercise_visit_proxy\',\n        \'exercise_stay_proxy\',\'forum_visit\',\'forum_stay\',\'forum_post\',\'quiz_stay_results_proxy\',\n        \'navigation_overview_visit\',\'navigation_overview_stay\',\'return_to_overview_rate\']\nPEND = [\'content_visit\',\'content_stay\',\'selfass_visit\',\'selfass_twice_wrong\',\'exercise_visit\',\'exercise_stay\',\n        \'ques_detail\',\'ques_overview\',\'ques_facts\',\'ques_concepts\',\'ques_graphics\',\'ques_text\',\'ques_interpret\',\n        \'ques_develop\',\'quiz_revisions\',\'quiz_stay_results\',\'navigation_overview_visit\',\'navigation_overview_stay\']\nFULL = \'OUTPUT/PRAPROSES/01_Praproses_main_LMS/05_dataset_behavioral_categorical_skala.csv\'\nRULE_FILES = {\'LMS\':\'OUTPUT/EKSPERIMEN/SEMANTIK/11_Interpretasi_Semantik_FSLSM/08_semantic_evidence_matrix.csv\',\n              \'PENDIKOM\':\'OUTPUT/EKSPERIMEN/HOLDOUT/PENDIKOM/70_30_groundtruth_semantic_rules.csv\'}\nDEFAULT_CONFIG = {\'bootstrap_repetitions\':10000,\'bootstrap_seed\':20261014,\n                  \'save_member_comparisons\':True,\'force\':False}\nMETRIC_NAMES = [\'accuracy\',\'balanced_accuracy\',\'macro_f1\']\n\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef digest(value):\n    return sha(json.dumps(clean(value),sort_keys=True,ensure_ascii=False,separators=(\',\',\':\'),allow_nan=False).encode(\'utf-8\'))\n\n\ndef clean(value):\n    if isinstance(value,dict): return {str(k):clean(v) for k,v in value.items()}\n    if isinstance(value,(list,tuple,np.ndarray)): return [clean(v) for v in value]\n    if isinstance(value,(np.integer,)): return int(value)\n    if isinstance(value,(float,np.floating)): return float(value) if np.isfinite(value) else None\n    if isinstance(value,np.bool_): return bool(value)\n    return value\n\n\ndef atomic(path, data):\n    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    temp = path.with_name(path.name+\'.\'+uuid.uuid4().hex[:8]+\'.tmp\')\n    temp.write_bytes(data); temp.replace(path)\n\n\ndef save_json(path, value):\n    atomic(path,json.dumps(clean(value),ensure_ascii=False,indent=2,allow_nan=False).encode(\'utf-8\'))\n\n\ndef save_csv(path, frame):\n    data = frame.to_csv(index=False,lineterminator=\'\\n\',float_format=\'%.17g\').encode(\'utf-8\')\n    atomic(path,gzip.compress(data,mtime=0) if str(path).endswith(\'.gz\') else data)\n\n\ndef read_checkpoint(path):\n    try:\n        obj = json.loads(Path(path).read_text(encoding=\'utf-8\'))\n        return obj if isinstance(obj,dict) else {}\n    except (OSError,ValueError,UnicodeError): return {}\n\n\nclass Files:\n    def __init__(self, root):\n        self.root = Path(root).resolve(); self.cache = {}\n\n    def path(self, relative):\n        path = (self.root / relative).resolve()\n        if not path.is_relative_to(self.root): raise ValueError(\'Lokasi di luar proyek: \'+relative)\n        return path\n\n    def read(self, relative):\n        path = self.path(relative); before = path.stat()\n        stamp = (before.st_size,before.st_mtime_ns,before.st_ctime_ns)\n        if relative in self.cache and self.cache[relative][0] == stamp: return self.cache[relative][1]\n        data = path.read_bytes(); after = path.stat()\n        if stamp != (after.st_size,after.st_mtime_ns,after.st_ctime_ns):\n            raise RuntimeError(\'Berkas berubah ketika dibaca: \'+relative)\n        self.cache[relative] = (stamp,data); return data\n\n    def hash(self, relative): return sha(self.read(relative))\n\n    def frame(self, relative, strings=False):\n        data = self.read(relative)\n        if relative.endswith(\'.gz\'): data = gzip.decompress(data)\n        return pd.read_csv(io.BytesIO(data),dtype=str if strings else None,\n                           keep_default_na=not strings,float_precision=\'round_trip\')\n\n\ndef runtime():\n    packages = {}\n    for name in [\'numpy\',\'pandas\',\'scipy\',\'sklearn\',\'matplotlib\']:\n        mod = sys.modules.get(name)\n        packages[name] = getattr(mod,\'__version__\',None) or importlib.metadata.version(\'scikit-learn\' if name==\'sklearn\' else name)\n    return {\'python\':platform.python_version(),\'packages\':packages}\n\n\ndef notebook_hash(files, stem, tag, variable):\n    nb = json.loads(files.read(\'PYTHON/\'+stem+\'.ipynb\').decode(\'utf-8-sig\')); code=[]\n    for cell in nb[\'cells\']:\n        if cell[\'cell_type\']!=\'code\': continue\n        source=\'\'.join(cell[\'source\'])\n        if tag in cell.get(\'metadata\',{}).get(\'tags\',[]):\n            tree=ast.parse(source)\n            if (len(tree.body)!=1 or not isinstance(tree.body[0],ast.Assign)\n                or len(tree.body[0].targets)!=1 or not isinstance(tree.body[0].targets[0],ast.Name)\n                or tree.body[0].targets[0].id!=variable or not isinstance(ast.literal_eval(tree.body[0].value),dict)):\n                raise RuntimeError(\'Sel konfigurasi hanya boleh berisi literal \'+variable)\n        else: code.append(source)\n    return digest(code)\n\n\ndef expected_rules(dataset):\n    rows=[]\n    def add(features,dim,low,high):\n        for f in features.split(): rows.append((f,dim,low,high))\n    if dataset==\'LMS\':\n        add(\'content_visit content_stay selfass_stay_proxy forum_visit quiz_stay_results_proxy\',DIMS[0],*POLES[0])\n        add(\'selfass_visit_proxy exercise_visit_proxy exercise_stay_proxy forum_post\',DIMS[0],*POLES[0][::-1])\n        add(\'content_visit content_stay\',DIMS[1],*POLES[1])\n        add(\'selfass_visit_proxy selfass_stay_proxy exercise_visit_proxy quiz_stay_results_proxy\',DIMS[1],*POLES[1][::-1])\n        add(\'content_visit forum_visit forum_stay forum_post\',DIMS[2],*POLES[2])\n        add(\'navigation_overview_visit navigation_overview_stay return_to_overview_rate\',DIMS[3],*POLES[3])\n    else:\n        add(\'selfass_visit selfass_twice_wrong exercise_visit exercise_stay\',DIMS[0],*POLES[0][::-1])\n        add(\'content_visit content_stay quiz_stay_results\',DIMS[0],*POLES[0])\n        add(\'selfass_visit exercise_visit ques_detail ques_facts quiz_revisions quiz_stay_results\',DIMS[1],*POLES[1][::-1])\n        add(\'content_visit content_stay ques_concepts ques_develop\',DIMS[1],*POLES[1])\n        add(\'ques_graphics\',DIMS[2],*POLES[2][::-1]); add(\'content_visit ques_text\',DIMS[2],*POLES[2])\n        add(\'ques_detail\',DIMS[3],*POLES[3][::-1])\n        add(\'ques_overview ques_interpret ques_develop navigation_overview_visit navigation_overview_stay\',DIMS[3],*POLES[3])\n    return sorted(rows)\n\n\ndef check_config(configuration):\n    if set(configuration)-set(DEFAULT_CONFIG): raise ValueError(\'Konfigurasi tidak dikenal.\')\n    cfg=dict(DEFAULT_CONFIG);cfg.update(configuration)\n    for key in [\'bootstrap_repetitions\',\'bootstrap_seed\']:\n        if isinstance(cfg[key],bool) or not isinstance(cfg[key],int) or cfg[key]<0: raise ValueError(key+\' tidak valid.\')\n    if cfg[\'bootstrap_repetitions\']<100: raise ValueError(\'Gunakan sedikitnya 100 bootstrap untuk diagnostik; default publikasi 10000.\')\n    for key in [\'save_member_comparisons\',\'force\']:\n        if not isinstance(cfg[key],bool): raise ValueError(key+\' harus True/False.\')\n    return cfg\n\n\ndef input_setup(files):\n    required = [UP+\'/PROTOKOL/02_konfigurasi.json\',UP_CONTROL+\'/status_eksekusi.json\',\n                FULL,*RULE_FILES.values(),\'LIB/pipeline_data.py\',\'LIB/pendikom_ils_labels.csv\',\n                \'OUTPUT/SPLIT/LMS/train_categorizer_state.json\',\n                \'OUTPUT/SPLIT/LMS/70_30_main_lms_train_only_thresholds.csv\',\n                \'OUTPUT/SPLIT/PENDIKOM/70_30_groundtruth_feature_audit.csv\',\n                \'OUTPUT/SPLIT/PENDIKOM/TEST/reference_ils.csv\',\n                \'PYTHON/\'+UPSTREAM+\'.ipynb\',\'PYTHON/\'+STAGE+\'.ipynb\']\n    for ds in [\'LMS\',\'PENDIKOM\']:\n        required += [f\'OUTPUT/SPLIT/{ds}/{part}/categorical.csv\' for part in [\'TRAIN\',\'TEST\']]\n    required += [f\'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{r:02d}.csv\' for r in range(30)]\n    missing=[p for p in required if not files.path(p).is_file()]\n    if missing: raise FileNotFoundError(\'Masukan belum tersedia. Selesaikan tahap 25 dan pertahankan hasil 01–24:\\n\'+\'\\n\'.join(missing[:20]))\n    status=json.loads(files.read(UP_CONTROL+\'/status_eksekusi.json\'))\n    if status.get(\'status\')!=\'selesai\' or status.get(\'full_protocol\') is not True or status.get(\'n_valid\')!=3272:\n        raise RuntimeError(\'Tahap 25 belum selesai dengan protokol lengkap dan seluruh run valid.\')\n    protocol=json.loads(files.read(UP+\'/PROTOKOL/02_konfigurasi.json\'))\n    if protocol[\'constants\'] != {\'FROZEN_CARE_CONFIG\':{\'candidate_rule\':\'separation\',\'candidate_budget\':16,\'min_gain\':0.0},\'K\':16,\'MAX_ITER\':100,\'SI_SAMPLE_SIZE\':2000}:\n        raise RuntimeError(\'Konfigurasi tahap 25 berbeda dari protokol yang ditetapkan.\')\n    source_code=protocol[\'code\']\n    for relative,checksum in source_code[\'project_libraries\'].items():\n        if files.hash(relative)!=checksum: raise RuntimeError(\'Pustaka sumber tahap 25 berubah: \'+relative)\n        required.append(relative)\n    if notebook_hash(files,UPSTREAM,\'initializer25_configuration\',\'KONFIGURASI_25\') != source_code[\'notebook_code\']:\n        raise RuntimeError(\'Kode notebook 25 berubah setelah eksperimen. Gunakan notebook sumber yang sesuai.\')\n    literals={}\n    nb=json.loads(files.read(\'PYTHON/\'+UPSTREAM+\'.ipynb\'))\n    for cell in nb[\'cells\']:\n        if cell[\'cell_type\']==\'code\':\n            for node in ast.parse(\'\'.join(cell[\'source\'])).body:\n                if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name) and isinstance(node.value,ast.Constant):\n                    literals[node.targets[0].id]=node.value.value\n    for name,var in [(\'initializer\',\'BK_MODES_SOURCE\'),(\'verification\',\'VERIFICATION_SOURCE\'),(\'comparison\',\'COMPARISON_SOURCE\')]:\n        if sha(literals[var].encode(\'utf-8\'))!=source_code[\'embedded_sources\'][name]: raise RuntimeError(\'Sumber tertanam tahap 25 tidak sesuai.\')\n    feature_state={}\n    for node in ast.parse(files.read(\'LIB/pipeline_data.py\').decode(\'utf-8-sig\')).body:\n        if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name) and node.targets[0].id in [\'MAIN_FEATURES\',\'PENDIKOM_FEATURES\']:\n            feature_state[node.targets[0].id]=ast.literal_eval(node.value)\n    if feature_state[\'MAIN_FEATURES\']!=MAIN or [x for x in feature_state[\'PENDIKOM_FEATURES\'] if x!=\'navigation_skip\']!=PEND:\n        raise RuntimeError(\'Urutan fitur tidak sesuai protokol.\')\n    audit=files.frame(\'OUTPUT/SPLIT/PENDIKOM/70_30_groundtruth_feature_audit.csv\',True)\n    if audit.loc[audit.used.str.lower().eq(\'true\'),\'feature\'].tolist()!=PEND: raise RuntimeError(\'Mask fitur train Pendikom berubah.\')\n    rules={}\n    for ds,path in RULE_FILES.items():\n        frame=files.frame(path,True)\n        cols=[\'feature\',\'dimension\',\'low_supports\',\'high_supports\']\n        if sorted(map(tuple,frame[cols].to_numpy())) != expected_rules(ds): raise RuntimeError(\'Evidence rules berbeda dari aturan utama: \'+ds)\n        if \'moderate_supports\' in frame and not frame.moderate_supports.eq(\'Neutral\').all(): raise RuntimeError(\'Evidence moderate harus netral.\')\n        rules[ds]=frame[cols]\n    watched={p:files.hash(p) for p in set(required) if not p.endswith(\'.ipynb\') and not p.endswith(\'status_eksekusi.json\')}\n    return protocol,rules,watched\n\n\ndef contexts(files):\n    def make(scope,ds,frame,features,idcol,source,rep=None,idx=None,test=None):\n        if frame[idcol].duplicated().any(): raise RuntimeError(\'Identitas peserta tidak unik: \'+scope)\n        X=frame[features].to_numpy(dtype=object)\n        if not np.isin(X,[\'1\',\'2\',\'3\']).all(): raise RuntimeError(\'Kategori di luar 1/2/3: \'+scope)\n        indices=np.arange(len(X),dtype=np.int64) if idx is None else idx\n        ids=frame[idcol].tolist();testids=test[idcol].tolist() if test is not None else []\n        if len(set(testids))!=len(testids) or set(ids)&set(testids): raise RuntimeError(\'Train/test tumpang tindih atau ID test berulang.\')\n        Xt=test[features].to_numpy(dtype=object) if test is not None else None\n        matrix=digest({\'features\':features,\'rows\':X.tolist(),\'ids\':ids,\'indices\':indices.tolist(),\n                       \'test_rows\':Xt.tolist() if Xt is not None else None,\'test_ids\':testids,\'classes\':None})\n        return {\'scope\':scope,\'dataset\':ds,\'X\':X,\'ids\':ids,\'indices\':indices,\'test_X\':Xt,\'test_ids\':testids,\n                \'features\':features,\'sources\':source,\'rep\':rep,\'matrix_sha256\':matrix,\'test_frame\':test}\n    frame=files.frame(FULL,True)\n    if frame.shape!=(6658,14) or frame.columns.tolist()!=[\'user\']+MAIN: raise RuntimeError(\'Matriks full LMS berbeda.\')\n    yield make(\'LMS_FULL\',\'LMS\',frame,MAIN,\'user\',[FULL])\n    for rep in range(30):\n        path=f\'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{rep:02d}.csv\'\n        idx=files.frame(path)[\'source_row_index\'].to_numpy(np.int64)\n        if len(idx)!=5326 or len(np.unique(idx))!=5326 or np.any(idx<0) or np.any(idx>=6658): raise RuntimeError(\'Indeks resample tidak valid.\')\n        yield make(\'LMS_RESAMPLING\',\'LMS\',frame.iloc[idx],MAIN,\'user\',[FULL,path],rep,idx)\n    for ds,ntr,nte,features,idcol in [(\'LMS\',4660,1998,MAIN,\'user\'),(\'PENDIKOM\',322,139,PEND,\'userid\')]:\n        paths=[f\'OUTPUT/SPLIT/{ds}/{part}/categorical.csv\' for part in [\'TRAIN\',\'TEST\']]\n        tr,te=[files.frame(p,True) for p in paths]\n        if len(tr)!=ntr or len(te)!=nte: raise RuntimeError(\'Ukuran train/test berbeda: \'+ds)\n        if \'split\' in tr and (not tr.split.str.lower().eq(\'train\').all() or not te.split.str.lower().eq(\'test\').all()): raise RuntimeError(\'Kolom split tidak sesuai.\')\n        yield make(ds+\'_TRAIN\',ds,tr,features,idcol,paths,test=te)\n\n\ndef job_identity(context,method,run):\n    return {\'scope\':context[\'scope\'],\'dataset\':context[\'dataset\'],\'rep\':context[\'rep\'],\'run\':run,\'method\':method}\n\n\ndef upstream_job(files,protocol,context,method,run):\n    identity=job_identity(context,method,run);key=digest(identity)[:24]\n    relative=UP_CONTROL+\'/JOBS/\'+key+\'.json\'\n    if not files.path(relative).is_file(): raise FileNotFoundError(\'Partisi tahap 25 belum tersedia: \'+str(identity))\n    record=json.loads(files.read(relative));checksum=record.pop(\'record_sha256\',\'\')\n    if digest(record)!=checksum or record.get(\'status\')!=\'selesai\' or record.get(\'identity\')!=identity:\n        raise RuntimeError(\'Catatan partisi tahap 25 tidak valid: \'+relative)\n    p=record[\'parameters\'];row=record[\'row\']\n    if record[\'signature\']!=digest({\'parameters\':p,\'code\':protocol[\'code\'],\'runtime\':protocol[\'runtime\']}): raise RuntimeError(\'Signature tahap 25 tidak sesuai protokol.\')\n    expected_seed=(6161600+context[\'rep\']) if context[\'rep\'] is not None else 4161600+run\n    if (p[\'matrix_sha256\']!=context[\'matrix_sha256\'] or p[\'k\']!=16 or p[\'seed\']!=expected_seed\n        or p[\'max_iter\']!=100 or p[\'care_config\']!=protocol[\'constants\'][\'FROZEN_CARE_CONFIG\']\n        or not p[\'needs_partition\'] or row.get(\'valid\') is not True or not row.get(\'partition_available\')):\n        raise RuntimeError(\'Isi matriks/parameter/partisi tahap 25 tidak sesuai: \'+str(identity))\n    watched={relative:files.hash(relative)}\n    for path,expected in record[\'outputs\'].items():\n        if not path.startswith(UP+\'/BANK/\'): raise RuntimeError(\'Lokasi bank tahap 25 tidak sesuai.\')\n        if files.hash(path)!=expected: raise RuntimeError(\'Artefak tahap 25 berubah. Pulihkan/jalankan ulang 25: \'+path)\n        watched[path]=expected\n    def find(filename):\n        found=[p for p in record[\'outputs\'] if p.endswith(\'/\'+filename)]\n        if len(found)!=1: raise RuntimeError(\'Artefak wajib tidak unik: \'+filename)\n        return found[0]\n    modes=files.frame(find(\'final_modes.csv\'),True)\n    if modes.columns.tolist()!=[\'cluster_id\']+context[\'features\'] or modes.cluster_id.astype(int).tolist()!=list(range(16)):\n        raise RuntimeError(\'Skema/urutan mode akhir tidak sesuai.\')\n    modes=modes[context[\'features\']].to_numpy(dtype=object)\n    if not np.isin(modes,[\'1\',\'2\',\'3\']).all(): raise RuntimeError(\'Kategori mode tidak valid.\')\n    assignment=files.frame(find(\'train_assignments.csv\'),True)\n    if assignment.identity.tolist()!=context[\'ids\'] or not np.array_equal(assignment.source_row_index.astype(np.int64),context[\'indices\']):\n        raise RuntimeError(\'Identitas/urutan anggota tidak sesuai matriks.\')\n    labels=assignment.cluster_id.to_numpy(int)\n    if not np.isin(labels,np.arange(16)).all() or (np.bincount(labels,minlength=16)==0).any(): raise RuntimeError(\'Keanggotaan cluster tidak valid.\')\n    cost=int(np.sum(context[\'X\']!=modes[labels]))\n    if cost!=int(row[\'final_cost\']): raise RuntimeError(\'Cost anggota/mode berbeda dari hasil tahap 25.\')\n    testlabels=None\n    if context[\'test_X\'] is not None:\n        test=files.frame(find(\'test_assignments.csv\'),True)\n        if test.identity.tolist()!=context[\'test_ids\']: raise RuntimeError(\'Urutan identitas test berbeda.\')\n        testlabels=test.cluster_id.to_numpy(int)\n        distance=np.zeros((len(testlabels),16),dtype=np.int64)\n        for j in range(len(context[\'features\'])): distance += context[\'test_X\'][:,j,None]!=modes[None,:,j]\n        if not np.array_equal(testlabels,np.argmin(distance,axis=1)) or not np.array_equal(test.distance.astype(int),distance.min(axis=1)):\n            raise RuntimeError(\'Assignment test berbeda dari mode train yang dibekukan.\')\n    return {\'record\':record,\'checksum\':checksum,\'watched\':watched,\'modes\':modes,\'labels\':labels,\'test_labels\':testlabels,\'identity\':identity}\n\n\ndef interpret(X,modes,labels,features,rules):\n    """Same mode-first rule and member-derived 11-unit magnitude as stages 11/13/22."""\n    sizes=np.bincount(labels,minlength=16);mapping=[];evidence=[];fpos={f:i for i,f in enumerate(features)}\n    prepared=[]\n    for dim,poles in zip(DIMS,POLES):\n        rr=rules.loc[rules.dimension.eq(dim)];cols=np.array([fpos[f] for f in rr.feature]);low=np.array([1 if v==poles[0] else -1 for v in rr.low_supports])\n        high=np.array([1 if v==poles[0] else -1 for v in rr.high_supports])\n        data=X[:,cols];vote=np.where(data==\'1\',low,np.where(data==\'3\',high,0))\n        mv=modes[:,cols];modevote=np.where(mv==\'1\',low,np.where(mv==\'3\',high,0))\n        prepared.append((rr,vote,modevote))\n    for cid in range(16):\n        item={\'cluster_id\':cid,\'cluster_name\':f\'C{cid+1:02d}\',\'cluster_size\':int(sizes[cid])};poles_selected=[];sources=[];conflicts=0\n        for di,(dim,poles,key) in enumerate(zip(DIMS,POLES,KEYS)):\n            rr,vote,modevote=prepared[di];mv=modevote[cid];nv=vote[labels==cid]\n            mp,mn,mz=[int(np.sum(mv==v)) for v in [1,-1,0]]\n            pos,neg,neu=[int(np.sum(nv==v)) for v in [1,-1,0]]\n            ms,ns=mp-mn,pos-neg;selected=1 if ms>0 else -1 if ms<0 else 1 if ns>0 else -1 if ns<0 else 0\n            source=\'MODE\' if ms else \'MEMBER\' if ns else \'BALANCED\';sources.append(source)\n            total=pos+neg+neu\n            units=min(11,max(0,int(np.floor(11*(pos+.5*neu)/total+.5)))) if total else None\n            raw=2*units-11 if units is not None else None;mag=abs(raw) if raw is not None else None\n            category=\'Balanced\' if mag in (1,3) else \'Moderate preference\' if mag in (5,7) else \'Strong preference\' if mag in (9,11) else \'Unresolved\'\n            pole=poles[0] if selected==1 else poles[1] if selected==-1 else None\n            short=ABBR[pole] if pole else \'Balanced\';poles_selected.append(pole)\n            conflict=bool(selected and ns and selected!=np.sign(ns));conflicts+=conflict\n            item[key]=short;item[key+\'_score\']=selected*mag if selected and mag is not None else np.nan\n            item[key+\'_category\']=category;item[key+\'_code\']=short+str(mag) if selected else \'Balanced\'\n            item[key+\'_resolved\']=bool(selected)\n            evidence.append({\'cluster_id\':cid,\'dimension\':dim,\'cluster_size\':int(sizes[cid]),\'mapped_features\':len(rr),\n                \'mode_positive\':mp,\'mode_negative\':mn,\'mode_neutral\':mz,\'mode_score\':ms,\'member_positive\':pos,\'member_negative\':neg,\n                \'member_neutral\':neu,\'member_score\':ns,\'selected_pole\':short,\'decision_source\':source,\n                \'positive_adjusted_evidence\':pos+.5*neu,\'negative_adjusted_evidence\':neg+.5*neu,\n                \'positive_11eq_units\':units,\'negative_11eq_units\':11-units if units is not None else None,\n                \'member_11eq_raw_score\':raw,\'preference_11eq_magnitude\':mag,\'preference_category\':category,\n                \'mode_member_consistency\':not conflict,\'strong_mode_member_conflict\':conflict and mag in (9,11)})\n        resolved=all(p is not None for p in poles_selected)\n        item.update(ls_class=LS_LOOKUP[tuple(poles_selected)] if resolved else \'AMBIGUOUS\',\n                    semantic_label=\'-\'.join(item[k] for k in KEYS),preference_signature=\'|\'.join(item[k+\'_code\'] for k in KEYS),\n                    mapping_status=\'RESOLVED\' if resolved else \'UNRESOLVED\',mode_dimensions=sources.count(\'MODE\'),\n                    member_dimensions=sources.count(\'MEMBER\'),balanced_dimensions=sources.count(\'BALANCED\'),mode_member_disagreements=conflicts)\n        mapping.append(item)\n    return pd.DataFrame(mapping),pd.DataFrame(evidence)\n\n\ndef profiles(mapping,labels,ids,indices=None):\n    cols=KEYS+[k+\'_score\' for k in KEYS]+[\'ls_class\',\'mapping_status\']\n    frame=mapping.set_index(\'cluster_id\').loc[labels,cols].reset_index()\n    frame.insert(0,\'identity\',ids)\n    if indices is not None: frame.insert(1,\'source_row_index\',indices)\n    return frame\n\n\ndef numeric_profiles(frame):\n    return np.column_stack([np.where(frame[k].eq(ABBR[p[0]]),1,np.where(frame[k].eq(ABBR[p[1]]),-1,0)) for k,p in zip(KEYS,POLES)])\n\n\ndef cache_read(files,path,signature,force=False):\n    record=read_checkpoint(path)\n    checksum=record.pop(\'record_sha256\',\'\')\n    try: valid=digest(record)==checksum\n    except (TypeError,ValueError): valid=False\n    if force or not valid or record.get(\'signature\')!=signature or record.get(\'status\')!=\'selesai\': return None\n    if not all(files.path(p).is_file() and files.hash(p)==h for p,h in record.get(\'outputs\',{}).items()): return None\n    return record\n\n\ndef cache_write(files,path,signature,outputs,extra):\n    record={\'status\':\'selesai\',\'signature\':signature,\'outputs\':{str(p.relative_to(files.root)).replace(\'\\\\\',\'/\'):sha(p.read_bytes()) for p in outputs},**extra}\n    record[\'record_sha256\']=digest(record);save_json(path,record);return record\n\n\ndef semantic_job(files,output,control,context,upstream,rules,code_state,cfg):\n    identity=upstream[\'identity\'];key=digest(identity)[:24]\n    signature=digest({\'code\':code_state,\'upstream_record_sha256\':upstream[\'checksum\'],\n                      \'matrix_sha256\':context[\'matrix_sha256\'],\'evidence\':rules.to_dict(\'records\'),\'rule\':\'mode-first-member-tie-11eq-v1\'})\n    checkpoint=control/\'PROFIL\'/f\'{key}.json\'\n    record=cache_read(files,checkpoint,signature,cfg[\'force\'])\n    folder=output/\'BANK\'/identity[\'scope\']/identity[\'dataset\']/(\'r%02d\' % (identity[\'rep\'] if identity[\'rep\'] is not None else identity[\'run\']))/re.sub(r\'[^A-Za-z0-9]+\',\'_\',identity[\'method\'].replace(\'++\',\'_PP\')).strip(\'_\')\n    mapping_path=folder/\'01_cluster_profiles.csv\';evidence_path=folder/\'02_cluster_dimension_evidence.csv\'\n    if record:\n        mapping=files.frame(str(mapping_path.relative_to(files.root)).replace(\'\\\\\',\'/\'));evidence=files.frame(str(evidence_path.relative_to(files.root)).replace(\'\\\\\',\'/\'))\n        disposition=\'CACHE\'\n    else:\n        mapping,evidence=interpret(context[\'X\'],upstream[\'modes\'],upstream[\'labels\'],context[\'features\'],rules)\n        save_csv(mapping_path,mapping);save_csv(evidence_path,evidence);paths=[mapping_path,evidence_path]\n        if context[\'test_X\'] is not None:\n            for filename,frame in [(\'03_train_targets.csv\',profiles(mapping,upstream[\'labels\'],context[\'ids\'],context[\'indices\'])),\n                                   (\'04_test_targets.csv\',profiles(mapping,upstream[\'test_labels\'],context[\'test_ids\']))]:\n                path=folder/filename;save_csv(path,frame);paths.append(path)\n        record=cache_write(files,checkpoint,signature,paths,{\'identity\':identity,\'upstream_artifacts\':upstream[\'watched\']})\n        disposition=\'MAP\'\n    weighted=sum(mapping.cluster_size*mapping.mapping_status.ne(\'RESOLVED\'))/len(context[\'X\'])\n    summary={**identity,\'n_learners\':len(context[\'X\']),\'n_clusters\':16,\'n_resolved_profiles\':mapping.loc[mapping.mapping_status.eq(\'RESOLVED\'),\'ls_class\'].nunique(),\n             \'unresolved_learner_rate\':weighted,\'mode_member_conflicts\':int((~evidence.mode_member_consistency.astype(bool)).sum()),\n             \'strong_conflicts\':int(evidence.strong_mode_member_conflict.astype(bool).sum()),\'mapping_file\':str(mapping_path.relative_to(files.root)).replace(\'\\\\\',\'/\'),\n             \'evidence_file\':str(evidence_path.relative_to(files.root)).replace(\'\\\\\',\'/\')}\n    return mapping,evidence,record,summary,disposition\n\n\ndef count_metrics(counts):\n    """2 x 3 confusion counts; column 3 is abstention, never dropped."""\n    a,b,c,d,e,f=np.moveaxis(np.asarray(counts,float),-1,0)\n    pos=a+b+c;neg=d+e+f;total=pos+neg\n    with np.errstate(divide=\'ignore\',invalid=\'ignore\'):\n        accuracy=(a+e)/total;ba=.5*(a/pos+e/neg)\n        denp=2*a+d+b+c;denn=2*e+b+d+f\n        fp=np.divide(2*a,denp,out=np.zeros_like(a),where=denp>0)\n        fn=np.divide(2*e,denn,out=np.zeros_like(e),where=denn>0)\n    return np.stack([accuracy,ba,.5*(fp+fn)],axis=-1)\n\n\ndef confusion_masks(truth,prediction):\n    # prediction: run x learner x dimension; output learner x run x dimension x six cells.\n    t=truth[None,:,:];p=prediction\n    conditions=[(t==1)&(p==1),(t==1)&(p==-1),(t==1)&(p==0),\n                (t==-1)&(p==1),(t==-1)&(p==-1),(t==-1)&(p==0)]\n    return np.stack(conditions,axis=-1).transpose(1,0,2,3).astype(float)\n\n\ndef ils_reference(files,context):\n    path=\'OUTPUT/SPLIT/PENDIKOM/TEST/reference_ils.csv\';ref=files.frame(path,True)\n    test=context[\'test_frame\']\n    if ref.userid.tolist()!=context[\'test_ids\'] or ref.courseid.tolist()!=test.courseid.tolist(): raise RuntimeError(\'Reference ILS tidak sejajar dengan peserta test.\')\n    original=files.frame(\'LIB/pendikom_ils_labels.csv\',True).set_index([\'userid\',\'courseid\'])\n    original=original.loc[list(zip(ref.userid,ref.courseid))].reset_index()\n    cols=[\'ils_\'+k+\'_code\' for k in KEYS]+[\'ils_ls_class\']\n    if not original[cols].reset_index(drop=True).equals(ref[cols].reset_index(drop=True)) or not original.split.str.lower().eq(\'test\').all():\n        raise RuntimeError(\'Reference ILS test berbeda dari label independen yang dibekukan.\')\n    truth=[]\n    for key,poles in zip(KEYS,POLES):\n        column=ref[\'ils_\'+key+\'_code\'];first=column.str.startswith(ABBR[poles[0]]);second=column.str.startswith(ABBR[poles[1]])\n        if not (first^second).all(): raise RuntimeError(\'Label arah ILS tidak dikenal.\')\n        values=np.where(first,1,-1)\n        if len(np.unique(values))!=2: raise RuntimeError(\'Dua kutub reference harus terwakili untuk metrik BA.\')\n        truth.append(values)\n    return ref,np.column_stack(truth)\n\n\ndef evaluate_ils(files,output,control,predictions,reference,truth,code_state,cfg):\n    folder=output/\'ILS\';checkpoint=control/\'evaluasi_ils.json\'\n    signature=digest({\'code\':code_state,\'truth\':truth.tolist(),\'ids\':reference.userid.tolist(),\n        \'predictions\':{m:sha(np.asarray(predictions[m],dtype=np.int8).tobytes()) for m in METHODS},\n        \'bootstrap_repetitions\':cfg[\'bootstrap_repetitions\'],\'bootstrap_seed\':cfg[\'bootstrap_seed\'],\n        \'unit\':\'learner; all four dimensions and all runs/methods remain paired\'})\n    saved=cache_read(files,checkpoint,signature,cfg[\'force\'])\n    if saved:\n        print(\'Evaluasi ILS/CI: gunakan hasil yang isinya terverifikasi sama.\',flush=True)\n        return files.frame(str((folder/\'04_ringkasan_overall.csv\').relative_to(files.root)).replace(\'\\\\\',\'/\')),True\n    from threadpoolctl import threadpool_limits\n    n=len(truth);nr=cfg[\'bootstrap_repetitions\'];rng=np.random.default_rng(cfg[\'bootstrap_seed\'])\n    masks={m:confusion_masks(truth,np.asarray(predictions[m],dtype=np.int8)) for m in METHODS}\n    counts={m:masks[m].sum(axis=0) for m in METHODS}\n    points={m:count_metrics(counts[m]) for m in METHODS}\n    boot=np.empty((nr,len(METHODS),5,3),dtype=np.float64)\n    valid=np.zeros(nr,dtype=bool)\n    print(\'CI ILS:\',nr,\'bootstrap berpasangan pada\',n,\'peserta.\',flush=True)\n    with threadpool_limits(limits=1):\n        for start in range(0,nr,250):\n            stop=min(start+250,nr);weights=rng.multinomial(n,np.full(n,1/n),size=stop-start).astype(float)\n            positive=weights@(truth==1);negative=n-positive\n            valid[start:stop]=((positive>0)&(negative>0)).all(axis=1)\n            for mi,method in enumerate(METHODS):\n                source=masks[method];shape=source.shape\n                values=(weights@source.reshape(n,-1)).reshape(stop-start,shape[1],4,6)\n                metrics=count_metrics(values).mean(axis=1)\n                boot[start:stop,mi,:4,:]=metrics;boot[start:stop,mi,4,:]=metrics.mean(axis=1)\n    if valid.sum()<100: raise RuntimeError(\'Bootstrap valid kurang dari 100; interval belum dapat dihitung.\')\n    boot=boot[valid];paths=[];raw_rows=[];cm_rows=[];ref_rows=[];summary=[];overall=[];pred_rows=[]\n    actual_counts=np.stack([(truth==1).sum(axis=0),(truth==-1).sum(axis=0)],axis=-1)\n    for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n        for ci,pole in enumerate(poles): ref_rows.append({\'dimension\':key,\'reference_pole\':pole,\'n\':int(actual_counts[di,ci]),\'n_test\':n})\n    for mi,method in enumerate(METHODS):\n        arr=np.asarray(predictions[method],dtype=np.int8);point=points[method];nrun=len(arr)\n        for run in range(nrun):\n            for li,identity in enumerate(reference.userid):\n                row={\'method\':method,\'run\':run,\'seed\':4161600+run,\'identity\':identity,\'courseid\':reference.courseid.iloc[li],\n                     \'reference_ls_class\':reference.ils_ls_class.iloc[li]}\n                full=[]\n                for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n                    row[\'reference_\'+key]=poles[0] if truth[li,di]==1 else poles[1]\n                    value=arr[run,li,di];pole=poles[0] if value==1 else poles[1] if value==-1 else None\n                    row[\'predicted_\'+key]=pole or \'UNRESOLVED\';full.append(pole)\n                row[\'predicted_ls_class\']=LS_LOOKUP[tuple(full)] if all(full) else \'AMBIGUOUS\';pred_rows.append(row)\n            for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n                cc=counts[method][run,di];a,b,c,d,e,f=cc;resolved=n-c-f\n                raw_rows.append({\'method\':method,\'run\':run,\'dimension\':key,\'n_test\':n,\'reference_first\':int(actual_counts[di,0]),\n                                 \'reference_second\':int(actual_counts[di,1]),\'n_resolved\':int(resolved),\'n_unresolved\':int(c+f),\n                                 \'coverage\':resolved/n,\'resolved_only_accuracy\':(a+e)/resolved if resolved else np.nan,\n                                 **dict(zip(METRIC_NAMES,point[run,di]))})\n                for ci,num in enumerate(cc):\n                    cm_rows.append({\'method\':method,\'run\':run,\'dimension\':key,\'reference_pole\':poles[ci//3],\n                                    \'predicted_pole\':(poles+(\'UNRESOLVED\',))[ci%3],\'n\':int(num)})\n        for di,key in enumerate(KEYS+[\'mean_four_dimensions\']):\n            values=point[:,di,:] if di<4 else point.mean(axis=1)\n            row={\'method\':method,\'dimension\':key,\'n_runs\':nrun,\'n_test\':n,\n                 \'bootstrap_valid\':int(valid.sum()),\'bootstrap_requested\':nr,\n                 \'coverage_mean\':float((arr[:,:,di]!=0).mean()) if di<4 else float((arr!=0).mean()),\n                 \'complete_profile_coverage\':float((arr!=0).all(axis=2).mean())}\n            for ji,metric in enumerate(METRIC_NAMES):\n                low,high=np.quantile(boot[:,mi,di,ji],[.025,.975])\n                row[metric+\'_mean\']=float(values[:,ji].mean());row[metric+\'_sd_runs\']=float(values[:,ji].std(ddof=1)) if nrun>1 else np.nan\n                row[metric+\'_ci95_low\']=low;row[metric+\'_ci95_high\']=high\n            (summary if di<4 else overall).append(row)\n    differences=[];care=METHODS.index(\'CARE-KM\')\n    for mi,method in enumerate(METHODS):\n        if method==\'CARE-KM\': continue\n        for di,key in enumerate(KEYS+[\'mean_four_dimensions\']):\n            for ji,metric in enumerate(METRIC_NAMES):\n                delta=boot[:,care,di,ji]-boot[:,mi,di,ji];lo,hi=np.quantile(delta,[.025,.975])\n                carepoint=points[\'CARE-KM\'][:,di,ji].mean() if di<4 else points[\'CARE-KM\'][:,:,ji].mean()\n                basepoint=points[method][:,di,ji].mean() if di<4 else points[method][:,:,ji].mean()\n                row={\'baseline\':method,\'dimension\':key,\'metric\':metric,\'care_minus_baseline\':carepoint-basepoint,\n                     \'paired_ci95_low\':lo,\'paired_ci95_high\':hi,\'positive_favors\':\'CARE-KM\',\'n_learners\':n,\n                     \'interval\':\'paired learner percentile bootstrap; individual 95% intervals\'}\n                if di==4 and metric==\'balanced_accuracy\':\n                    low,high=np.quantile(delta,[.05/(2*7),1-.05/(2*7)])\n                    row.update(primary_family=\'seven overall mean-BA contrasts\',bonferroni_family95_low=low,bonferroni_family95_high=high)\n                differences.append(row)\n    outputs=[(\'01_prediksi_per_peserta.csv\',pd.DataFrame(pred_rows)),(\'02_metrik_per_dimensi_per_run.csv\',pd.DataFrame(raw_rows)),\n             (\'03_ringkasan_per_dimensi.csv\',pd.DataFrame(summary)),(\'04_ringkasan_overall.csv\',pd.DataFrame(overall)),\n             (\'05_confusion_matrices.csv\',pd.DataFrame(cm_rows)),(\'07_perbedaan_berpasangan_vs_care.csv\',pd.DataFrame(differences)),\n             (\'08_reference_counts.csv\',pd.DataFrame(ref_rows))]\n    for filename,frame in outputs:\n        path=folder/filename;save_csv(path,frame);paths.append(path)\n    buffer=io.BytesIO();np.savez_compressed(buffer,metrics=boot,methods=np.asarray(METHODS),dimensions=np.asarray(KEYS+[\'mean_four_dimensions\']),\n                                          metric_names=np.asarray(METRIC_NAMES),valid_draws=valid,seed=cfg[\'bootstrap_seed\'])\n    path=folder/\'06_bootstrap_paired_metrics.npz\';atomic(path,buffer.getvalue());paths.append(path)\n    path=folder/\'09_protokol_evaluasi.json\'\n    save_json(path,{\'primary_metric\':\'mean balanced accuracy across four dimensions\',\'n_learners\':n,\'n_bootstrap_valid\':int(valid.sum()),\n        \'seed\':cfg[\'bootstrap_seed\'],\'unit\':\'learner; all dimensions, runs, and methods paired\',\n        \'stochastic_summary\':\'metric per run, then arithmetic mean; no best-run selection\',\n        \'interval_scope\':\'conditional on frozen training split, modes, mapping, and saved initialization runs\',\n        \'unresolved\':\'included as incorrect/abstention; two reference poles used for macro-F1; coverage separately reported\',\n        \'multiplicity\':\'individual paired CIs descriptive; primary seven overall mean-BA contrasts also have Bonferroni percentile family intervals\',\n        \'semantic_scope\':\'direction correspondence; behavioral 11-unit magnitudes are not questionnaire scores\'})\n    paths.append(path)\n    cache_write(files,checkpoint,signature,paths,{\'n_test\':n,\'n_methods\':8,\'bootstrap_valid\':int(valid.sum())})\n    return pd.DataFrame(overall),False\n\n\ndef stability_job(files,output,control,context,upstream,mapping,reference,code_state,cfg):\n    from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score,adjusted_mutual_info_score\n    method=upstream[\'identity\'][\'method\'];rep=context[\'rep\'];identity=upstream[\'identity\'];key=digest(identity)[:24]\n    signature=digest({\'code\':code_state,\'upstream\':upstream[\'checksum\'],\'mapping\':mapping.to_dict(\'records\'),\n        \'reference\':reference[\'signature\'],\'save_members\':cfg[\'save_member_comparisons\']})\n    path=control/\'STABILITAS\'/f\'{key}.json\';cached=cache_read(files,path,signature,cfg[\'force\'])\n    if cached: return cached[\'summary\'],\'CACHE\'\n    idx=context[\'indices\'];current=profiles(mapping,upstream[\'labels\'],context[\'ids\'],idx)\n    ref=reference[\'profiles\'].iloc[idx].reset_index(drop=True)\n    if ref.identity.tolist()!=context[\'ids\']: raise RuntimeError(\'Peserta resample tidak sejajar dengan full reference initializer.\')\n    truth=numeric_profiles(ref);pred=numeric_profiles(current);paired=(truth!=0)&(pred!=0);agreement=paired&(truth==pred)\n    complete=agreement.all(axis=1)\n    row={**identity,\'n_learners\':len(idx),\'reference_full_run\':0,\'reference_seed\':4161600,\n        \'mean_direction_agreement\':float(agreement.mean()),\'all_four_direction_agreement\':float(complete.mean()),\'base_ls_agreement\':float(complete.mean()),\n        \'reference_complete_coverage\':float((truth!=0).all(axis=1).mean()),\'resample_complete_coverage\':float((pred!=0).all(axis=1).mean()),\n        \'unresolved_learner_rate\':float((pred==0).any(axis=1).mean()),\'joint_resolved_dimensional_coverage\':float(paired.mean()),\n        \'n_resolved_profiles\':int(current.loc[current.mapping_status.eq(\'RESOLVED\'),\'ls_class\'].nunique()),\n        \'ari\':float(adjusted_rand_score(ref.cluster_id,current.cluster_id)),\n        \'nmi\':float(normalized_mutual_info_score(ref.cluster_id,current.cluster_id)),\n        \'ami\':float(adjusted_mutual_info_score(ref.cluster_id,current.cluster_id))}\n    for di,keydim in enumerate(KEYS):\n        row[keydim+\'_agreement\']=float(agreement[:,di].mean())\n        row[keydim+\'_joint_resolved_coverage\']=float(paired[:,di].mean())\n        row[keydim+\'_resolved_only_agreement\']=float(agreement[:,di].sum()/paired[:,di].sum()) if paired[:,di].sum() else np.nan\n    refscore=ref[[k+\'_score\' for k in KEYS]].to_numpy(float);score=current[[k+\'_score\' for k in KEYS]].to_numpy(float)\n    row[\'mean_absolute_signed_score_change\']=float(np.nanmean(np.abs(score-refscore))) if paired.any() else np.nan\n    paths=[]\n    if cfg[\'save_member_comparisons\']:\n        frame=pd.DataFrame({\'identity\':context[\'ids\'],\'source_row_index\':idx,\'reference_cluster_id\':ref.cluster_id,\n                            \'resample_cluster_id\':current.cluster_id,\'reference_ls_class\':ref.ls_class,\'resample_ls_class\':current.ls_class,\n                            \'all_four_agreement\':complete})\n        for di,k in enumerate(KEYS):\n            frame[k+\'_reference\']=ref[k];frame[k+\'_resample\']=current[k];frame[k+\'_agreement\']=agreement[:,di]\n        member_path=output/\'LMS_STABILITAS/ANGGOTA\'/(\'rep_%02d_\' % rep+re.sub(r\'[^A-Za-z0-9]+\',\'_\',method.replace(\'++\',\'_PP\'))+\'.csv.gz\')\n        save_csv(member_path,frame);paths.append(member_path)\n    cache_write(files,path,signature,paths,{\'summary\':clean(row),\'identity\':identity})\n    return row,\'CALCULATE\'\n\n\ndef summarize_stability(rows):\n    raw=pd.DataFrame(rows);summary=[]\n    metrics=[\'mean_direction_agreement\',\'all_four_direction_agreement\',\'base_ls_agreement\',\'unresolved_learner_rate\',\n             \'joint_resolved_dimensional_coverage\',\'n_resolved_profiles\',\'ari\',\'nmi\',\'ami\']+[k+\'_agreement\' for k in KEYS]\n    for method in ORDER:\n        frame=raw.loc[raw.method.eq(method)]\n        if len(frame)!=30 or frame.rep.nunique()!=30: raise RuntimeError(\'Stabilitas harus berisi 30 resample per metode.\')\n        row={\'method\':method,\'n_resamples\':30}\n        for metric in metrics:\n            row[metric+\'_mean\']=frame[metric].mean();row[metric+\'_sd\']=frame[metric].std(ddof=1)\n            row[metric+\'_min\']=frame[metric].min();row[metric+\'_max\']=frame[metric].max()\n        summary.append(row)\n    paired=[];care=raw.loc[raw.method.eq(\'CARE-KM\')].set_index(\'rep\')\n    for method in [m for m in ORDER if m!=\'CARE-KM\']:\n        other=raw.loc[raw.method.eq(method)].set_index(\'rep\').loc[care.index]\n        for metric in [\'mean_direction_agreement\',\'base_ls_agreement\',\'ari\']:\n            delta=care[metric]-other[metric]\n            paired.append({\'baseline\':method,\'metric\':metric,\'care_minus_baseline_mean\':delta.mean(),\'sd_matched_differences\':delta.std(ddof=1),\n                           \'wins\':int((delta>1e-12).sum()),\'ties\':int((delta.abs()<=1e-12).sum()),\'losses\':int((delta<-1e-12).sum()),\n                           \'interpretation\':\'descriptive matched overlapping resamples; not 30 independent cohorts\'})\n    return raw,pd.DataFrame(summary),pd.DataFrame(paired)\n\n\ndef figures(output,ils,stability):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    paths=[];folder=output/\'GAMBAR\';folder.mkdir(parents=True,exist_ok=True)\n    data=ils.set_index(\'method\').loc[ORDER];y=np.arange(len(data))\n    fig,ax=plt.subplots(figsize=(8.5,5))\n    values=data.balanced_accuracy_mean.to_numpy()*100\n    # Percentile CIs need not contain the point estimate; draw their actual endpoints.\n    low=data.balanced_accuracy_ci95_low.to_numpy()*100;high=data.balanced_accuracy_ci95_high.to_numpy()*100\n    ax.hlines(y,low,high,color=\'#333333\');ax.plot(values,y,\'o\',color=\'#185785\')\n    ax.set_yticks(y,ORDER);ax.invert_yaxis();ax.set_xlabel(\'Mean balanced accuracy (%)\');ax.set_xlim(0,100)\n    ax.set_title(\'Pendikom holdout: direct profiles and paired learner bootstrap CIs\');fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'01_ils_mean_balanced_accuracy.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig)\n    fig,ax=plt.subplots(figsize=(9,5));groups=[stability.loc[stability.method.eq(m),\'mean_direction_agreement\'].to_numpy()*100 for m in ORDER]\n    ax.boxplot(groups,positions=np.arange(1,9),widths=.55)\n    ax.set_xticks(np.arange(1,9),ORDER,rotation=30,ha=\'right\');ax.set_ylim(0,100)\n    ax.set_ylabel(\'Mean agreement of four directions (%)\');ax.set_title(\'LMS semantic stability across 30 matched 80% resamples\');fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'02_lms_semantic_stability.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig)\n    return paths\n\n\ndef unchanged(files,watched,code_checks):\n    for path,checksum in watched.items():\n        if files.hash(path)!=checksum: raise RuntimeError(\'Masukan berubah selama eksekusi: \'+path)\n    for stem,tag,var,expected in code_checks:\n        if notebook_hash(files,stem,tag,var)!=expected: raise RuntimeError(\'Kode notebook berubah selama eksekusi: \'+stem)\n\n\ndef run_semantics(root,configuration,embedded_source):\n    root=Path(root).expanduser().resolve();files=Files(root);cfg=check_config(configuration)\n    protocol,rules,watched=input_setup(files)\n    own_hash=notebook_hash(files,STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\')\n    code_checks=[(UPSTREAM,\'initializer25_configuration\',\'KONFIGURASI_25\',protocol[\'code\'][\'notebook_code\']),\n                 (STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\',own_hash)]\n    code_state={\'implementation_sha256\':sha(embedded_source.encode(\'utf-8\')),\'notebook_code\':own_hash,\'runtime\':runtime()}\n    output=root/\'OUTPUT/TAMBAHAN\'/STAGE;control=output/\'KONTROL\'\n    output.mkdir(parents=True,exist_ok=True);control.mkdir(parents=True,exist_ok=True)\n    save_json(output/\'PROTOKOL/01_konfigurasi.json\',{\'configuration\':cfg,\'code\':code_state,\'upstream_protocol\':protocol,\n        \'semantic_rule\':\'mode-first; member tie-break; unchanged member-derived 11-unit magnitude\',\n        \'primary_metric\':\'Pendikom mean balanced accuracy\',\'stability_reference\':\'own initializer full run 0; no reference chosen by outcomes\',\n        \'method_count\':8,\'mapping_jobs_expected\':432,\'stability_jobs_expected\':240,\'bootstrap_unit\':\'learner; all methods/runs/dimensions paired\',\n        \'unknown_vs_balanced\':\'UNRESOLVED direction differs from Balanced preference magnitude {1,3}\',\n        \'no_new_clustering\':True})\n    for ds in [\'LMS\',\'PENDIKOM\']: save_csv(output/\'PROTOKOL\'/(\'02_evidence_\'+ds+\'.csv\'),rules[ds])\n    counts={\'CACHE\':0,\'MAP\':0};stability_counts={\'CACHE\':0,\'CALCULATE\':0};rows=[];artifacts=[];stable=[];references={}\n    predictions={m:[] for m in METHODS};reference=None;truth=None;all_watched=dict(watched)\n    try:\n        for ci,context in enumerate(contexts(files),1):\n            unchanged(files,watched,code_checks)\n            print(f"[{ci}/33] {context[\'scope\']} | rep={context[\'rep\']} | n={len(context[\'X\'])}",flush=True)\n            for method in METHODS:\n                runs=15 if method in STOCHASTIC and context[\'scope\']!=\'LMS_RESAMPLING\' else 1\n                for run in range(runs):\n                    upstream=upstream_job(files,protocol,context,method,run);all_watched.update(upstream[\'watched\'])\n                    mapping,evidence,record,summary,disposition=semantic_job(files,output,control,context,upstream,rules[context[\'dataset\']],code_state,cfg)\n                    counts[disposition]+=1;rows.append(summary)\n                    artifacts.append({**upstream[\'identity\'],\'upstream_record_sha256\':upstream[\'checksum\'],\n                                      \'files\':json.dumps(list(record[\'outputs\']),ensure_ascii=False),\n                                      \'upstream_files\':json.dumps(list(upstream[\'record\'][\'outputs\']),ensure_ascii=False)})\n                    if context[\'scope\']==\'LMS_FULL\' and run==0:\n                        frame=profiles(mapping,upstream[\'labels\'],context[\'ids\'],context[\'indices\'])\n                        references[method]={\'profiles\':frame,\'signature\':digest({\'upstream\':upstream[\'checksum\'],\'mapping\':mapping.to_dict(\'records\')})}\n                    if context[\'scope\']==\'LMS_RESAMPLING\':\n                        row,dispo=stability_job(files,output,control,context,upstream,mapping,references[method],code_state,cfg)\n                        stable.append(row);stability_counts[dispo]+=1\n                    if context[\'scope\']==\'PENDIKOM_TRAIN\':\n                        if reference is None: reference,truth=ils_reference(files,context)\n                        predictions[method].append(numeric_profiles(profiles(mapping,upstream[\'test_labels\'],context[\'test_ids\'])))\n            save_json(control/\'status_eksekusi.json\',{\'status\':\'berjalan\',\'completed_mapping_jobs\':len(rows),\'mapping_counts\':counts,\'last_context\':context[\'scope\']})\n        if len(rows)!=432 or len(stable)!=240: raise RuntimeError(\'Jumlah partisi/stabilitas belum lengkap.\')\n        unchanged(files,all_watched,code_checks)\n        ils,ils_cached=evaluate_ils(files,output,control,predictions,reference,truth,code_state,cfg)\n        raw,stability,paired=summarize_stability(stable)\n        save_csv(output/\'01_ringkasan_semua_partisi.csv\',pd.DataFrame(rows))\n        save_csv(output/\'02_peta_artefak.csv\',pd.DataFrame(artifacts))\n        save_csv(output/\'LMS_STABILITAS/01_metrik_per_resample.csv\',raw)\n        save_csv(output/\'LMS_STABILITAS/02_ringkasan_8_initializer.csv\',stability)\n        save_csv(output/\'LMS_STABILITAS/03_perbedaan_berpasangan_deskriptif.csv\',paired)\n        save_json(output/\'LMS_STABILITAS/04_protokol_stabilitas.json\',{\n            \'reference\':\'own-method LMS full run 0 (seed 4161600), declared in stage 25\',\n            \'resampling\':\'30 fixed matched 80% samples; overlapping learners\',\n            \'inference\':\'descriptive mean/SD/range and matched differences; no independent-cohort p-values\',\n            \'unresolved\':\'never counted as correct direction agreement; conditional agreement includes paired coverage\',\n            \'assignment_source\':\'stage 25 bank; no reclustering\', \'compressed_members\':\'CSV inside gzip; learner IDs, profiles and agreements are auditable\'})\n        summary=pd.DataFrame(rows)\n        save_csv(output/\'LMS_FULL/01_keragaman_profil_per_run.csv\',summary.loc[summary.scope.eq(\'LMS_FULL\')])\n        paper=ils.set_index(\'method\').loc[ORDER].reset_index()\n        for metric in METRIC_NAMES:\n            paper[metric+\'_mean_ci95\']=paper.apply(lambda r: f"{100*r[metric+\'_mean\']:.2f} [{100*r[metric+\'_ci95_low\']:.2f}, {100*r[metric+\'_ci95_high\']:.2f}]",axis=1)\n        save_csv(output/\'TABEL/01_initializer_ils.csv\',paper[[\'method\',\'n_runs\']+[m+\'_mean_ci95\' for m in METRIC_NAMES]+[\'coverage_mean\',\'complete_profile_coverage\']])\n        save_csv(output/\'TABEL/02_initializer_stabilitas.csv\',stability[[\'method\',\'n_resamples\',\'mean_direction_agreement_mean\',\'mean_direction_agreement_sd\',\n            \'base_ls_agreement_mean\',\'base_ls_agreement_sd\',\'unresolved_learner_rate_mean\',\'n_resolved_profiles_mean\',\'ari_mean\']])\n        # Render only when figure inputs/code changed or a figure is missing/tampered.\n        fig_signature=digest({\'code\':code_state,\'ils\':sha((output/\'ILS/04_ringkasan_overall.csv\').read_bytes()),\n                              \'stability\':sha((output/\'LMS_STABILITAS/01_metrik_per_resample.csv\').read_bytes())})\n        fig_cache=cache_read(files,control/\'gambar.json\',fig_signature,cfg[\'force\'])\n        if not fig_cache: cache_write(files,control/\'gambar.json\',fig_signature,figures(output,ils,raw),{})\n        unchanged(files,all_watched,code_checks)\n        save_json(output/\'PROTOKOL/03_manifest_masukan.json\',all_watched)\n        status={\'status\':\'selesai\',\'completed_mapping_jobs\':432,\'completed_stability_jobs\':240,\'method_count\':8,\n                \'mapping_counts\':counts,\'stability_counts\':stability_counts,\'ils_metrics_cached\':ils_cached,\n                \'full_protocol\':cfg[\'bootstrap_repetitions\']>=10000,\'n_ils_test\':139,\n                \'new_clustering_runs\':0,\'output\':str(output),\'finished_utc\':datetime.now(timezone.utc).isoformat()}\n        # Manifest includes stage-26 scientific deliverables, not mutable control logs.\n        outputs={str(p.relative_to(root)).replace(\'\\\\\',\'/\'):sha(p.read_bytes()) for p in output.rglob(\'*\') if p.is_file() and control not in p.parents}\n        save_json(control/\'manifest_keluaran.json\',outputs);save_json(control/\'status_eksekusi.json\',status)\n        print(\'Tahap 26 selesai. Partisi:\',counts,\'| Stabilitas:\',stability_counts,\'| ILS cache:\',ils_cached,flush=True)\n        print(\'Hasil:\',output)\n        return {\'ils\':ils,\'stability\':stability,\'status\':status,\'output\':str(output)}\n    except BaseException as error:\n        save_json(control/\'status_eksekusi.json\',{\'status\':\'terhenti\',\'completed_mapping_jobs\':len(rows),\n                  \'mapping_counts\':counts,\'error\':type(error).__name__+\': \'+str(error),\n                  \'resume\':\'jalankan notebook 26 kembali; cache diperiksa berdasarkan isi\'})\n        raise\n'

_s26 = types.ModuleType('_stage26_helpers')
exec(compile(HELPER26_SOURCE, '_stage26_helpers', 'exec'), _s26.__dict__)
_s26.EMBEDDED_SOURCE = HELPER26_SOURCE


In [4]:
FACTORY22_SOURCE = "def one_hot_encoder():\n    try:\n        return OneHotEncoder(handle_unknown='ignore', sparse_output=True)\n    except TypeError:\n        return OneHotEncoder(handle_unknown='ignore', sparse=True)\n\ndef categorical_nb():\n    try:\n        return CategoricalNB(alpha=1.0, min_categories=4)\n    except TypeError:\n        return CategoricalNB(alpha=1.0)\n\ndef build_external_classifiers():\n    # Frozen configuration used by the locked ground-truth Table 2.\n    return {\n        'NB': categorical_nb(),\n        'DT': DecisionTreeClassifier(random_state=SPLIT_SEED, class_weight='balanced'),\n        'SVM': Pipeline([('onehot', one_hot_encoder()), ('svc', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced'))]),\n    }\n"

SUPERVISED28_SOURCE = '"""Paired CDS/ILS supervision controls on the frozen Pendikom holdout."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport ast\nimport io\nimport json\nimport time\nimport types\nimport numpy as np\nimport pandas as pd\n\nSTAGE=\'28_Referensi_Supervised_ILS\'\nU26=\'OUTPUT/TAMBAHAN/26_Pengaruh_Initializer_Profil_Semantik\'\nU27=\'OUTPUT/TAMBAHAN/27_Perbandingan_Pembentukan_Profil\'\nFAMILIES=[\'NB\',\'DT\',\'SVM\']\nPATHS=[\'CARE_DIRECT\',\'BEHAVIORAL_DIRECT\',\'CDS_NB\',\'CDS_DT\',\'CDS_SVM\',\'ILS_NB\',\'ILS_DT\',\'ILS_SVM\',\'ILS_TRAIN_MAJORITY\']\nDEFAULT_CONFIG={\'force\':False}\n\n\ndef parse_notebook(files,stem):\n    nb=json.loads(files.read(\'PYTHON/\'+stem+\'.ipynb\').decode(\'utf-8-sig\'))\n    return \'\\n\\n\'.join(\'\'.join(c[\'source\']) for c in nb[\'cells\'] if c[\'cell_type\']==\'code\')\n\n\ndef literal_sources(files,stem):\n    result={}\n    for node in ast.parse(parse_notebook(files,stem)).body:\n        if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name) and isinstance(node.value,ast.Constant):\n            result[node.targets[0].id]=node.value.value\n    return result\n\n\ndef frame_codes(s,reference):\n    codes=[]\n    for key,poles in zip(s.KEYS,s.POLES):\n        x=reference[\'ils_\'+key+\'_code\'];a=x.str.startswith(s.ABBR[poles[0]]);b=x.str.startswith(s.ABBR[poles[1]])\n        if not (a^b).all():raise RuntimeError(\'Kode ILS tidak dikenal: \'+key)\n        codes.append(np.where(a,1,-1))\n    arr=np.column_stack(codes)\n    classes=[s.LS_LOOKUP[tuple(p[0] if v==1 else p[1] for v,p in zip(row,s.POLES))] for row in arr]\n    if reference.ils_ls_class.tolist()!=classes:raise RuntimeError(\'Kode Base LS dan arah ILS tidak konsisten.\')\n    return arr\n\n\ndef primary26(s,files,p26,ctx,up,rules):\n    signature=s.digest({\'code\':p26[\'code\'],\'upstream_record_sha256\':up[\'checksum\'],\'matrix_sha256\':ctx[\'matrix_sha256\'],\n        \'evidence\':rules.to_dict(\'records\'),\'rule\':\'mode-first-member-tie-11eq-v1\'})\n    path=files.path(U26+\'/KONTROL/PROFIL/\'+s.digest(up[\'identity\'])[:24]+\'.json\')\n    record=s.cache_read(files,path,signature)\n    if record is None or record.get(\'upstream_artifacts\')!=up[\'watched\']:raise RuntimeError(\'Target CDS tahap 26 tidak sesuai sumbernya.\')\n    def find(filename):\n        found=[p for p in record[\'outputs\'] if p.endswith(\'/\'+filename)]\n        if len(found)!=1:raise RuntimeError(\'Artefak target CDS tidak lengkap: \'+filename)\n        return found[0]\n    tr=files.frame(find(\'03_train_targets.csv\'),True);te=files.frame(find(\'04_test_targets.csv\'),True)\n    if tr.identity.tolist()!=ctx[\'ids\'] or te.identity.tolist()!=ctx[\'test_ids\'] or not tr.mapping_status.eq(\'RESOLVED\').all():\n        raise RuntimeError(\'Target CDS harus sejajar dan seluruh 322 train resolved untuk perbandingan utama.\')\n    watched={str(path.relative_to(files.root)).replace(\'\\\\\',\'/\'):files.hash(str(path.relative_to(files.root)).replace(\'\\\\\',\'/\')),**record[\'outputs\']}\n    return tr,te,watched\n\n\ndef prepare(s,files,factory_source):\n    required=[\'PYTHON/\'+STAGE+\'.ipynb\',\'PYTHON/22_Holdout_LMS_Pendikom.ipynb\',\n        \'LIB/pipeline_data.py\',\'LIB/pendikom_ils_labels.csv\',\'OUTPUT/SPLIT/PENDIKOM/70_30_groundtruth_feature_audit.csv\',\n        s.UP+\'/PROTOKOL/02_konfigurasi.json\',U26+\'/PROTOKOL/01_konfigurasi.json\',\n        U27+\'/PROTOKOL/01_konfigurasi.json\',U27+\'/KONTROL/status_eksekusi.json\',U27+\'/KONTROL/evaluasi_ils.json\',s.RULE_FILES[\'PENDIKOM\']]\n    required += [f\'OUTPUT/SPLIT/PENDIKOM/{part}/{name}\' for part in [\'TRAIN\',\'TEST\'] for name in [\'categorical.csv\',\'reference_ils.csv\']]\n    missing=[p for p in required if not files.path(p).is_file()]\n    if missing:raise FileNotFoundError(\'Masukan tahap 28 belum tersedia:\\n\'+\'\\n\'.join(missing))\n    status=json.loads(files.read(U27+\'/KONTROL/status_eksekusi.json\'))\n    if status.get(\'status\')!=\'selesai\' or status.get(\'n_unique_ils_pathways\')!=25 or status.get(\'full_protocol\') is not True:\n        raise RuntimeError(\'Selesaikan tahap 27 dengan protokol penuh terlebih dahulu.\')\n    p25=json.loads(files.read(s.UP+\'/PROTOKOL/02_konfigurasi.json\'));p26=json.loads(files.read(U26+\'/PROTOKOL/01_konfigurasi.json\'));p27=json.loads(files.read(U27+\'/PROTOKOL/01_konfigurasi.json\'))\n    if p26[\'upstream_protocol\']!=p25:raise RuntimeError(\'Protokol 25/26 tidak sama.\')\n    if p26[\'code\'][\'implementation_sha256\']!=s.sha(s.EMBEDDED_SOURCE.encode(\'utf-8\')) or p27[\'code\'][\'helper26_sha256\']!=p26[\'code\'][\'implementation_sha256\']:\n        raise RuntimeError(\'Versi helper/sumber tahap 26 tidak didukung.\')\n    checks=[(s.UPSTREAM,\'initializer25_configuration\',\'KONFIGURASI_25\',p25[\'code\'][\'notebook_code\']),\n        (s.STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\',p26[\'code\'][\'notebook_code\']),\n        (\'27_Perbandingan_Pembentukan_Profil\',\'profile27_configuration\',\'KONFIGURASI_27\',p27[\'code\'][\'notebook_code\'])]\n    for stem,tag,var,h in checks:\n        if s.notebook_hash(files,stem,tag,var)!=h:raise RuntimeError(\'Kode sumber berubah setelah eksperimen: \'+stem)\n    for path,h in p25[\'code\'][\'project_libraries\'].items():\n        if files.hash(path)!=h:raise RuntimeError(\'Pustaka sumber berubah: \'+path)\n        required.append(path)\n    v25=literal_sources(files,s.UPSTREAM)\n    for name,var in [(\'initializer\',\'BK_MODES_SOURCE\'),(\'verification\',\'VERIFICATION_SOURCE\'),(\'comparison\',\'COMPARISON_SOURCE\')]:\n        if s.sha(v25[var].encode(\'utf-8\'))!=p25[\'code\'][\'embedded_sources\'][name]:raise RuntimeError(\'Sumber tertanam 25 berubah.\')\n    v27=literal_sources(files,\'27_Perbandingan_Pembentukan_Profil\')\n    if s.sha(v27[\'PROFILE27_SOURCE\'].encode(\'utf-8\'))!=p27[\'code\'][\'implementation_sha256\']:raise RuntimeError(\'Sumber tertanam 27 berubah.\')\n    code22=parse_notebook(files,\'22_Holdout_LMS_Pendikom\');tree=ast.parse(code22)\n    names=[\'one_hot_encoder\',\'categorical_nb\',\'build_external_classifiers\']\n    actual=[n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name in names]\n    actual={n.name:n for n in actual};expected=ast.parse(factory_source)\n    if len(actual)!=3 or ast.dump(ast.Module(body=[actual[k] for k in names],type_ignores=[]))!=ast.dump(expected):\n        raise RuntimeError(\'Konfigurasi classifier skrip 22 berbeda dari konfigurasi pasangan tahap 28.\')\n    seeds=[ast.literal_eval(n.value) for n in tree.body if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id==\'SPLIT_SEED\' for t in n.targets)]\n    if seeds!=[161600]:raise RuntimeError(\'Seed source classifier berbeda.\')\n    library=ast.parse(files.read(\'LIB/pipeline_data.py\').decode(\'utf-8-sig\'))\n    features=next(ast.literal_eval(n.value) for n in library.body if isinstance(n,ast.Assign) and len(n.targets)==1 and isinstance(n.targets[0],ast.Name) and n.targets[0].id==\'PENDIKOM_FEATURES\')\n    if [f for f in features if f!=\'navigation_skip\']!=s.PEND:raise RuntimeError(\'Daftar fitur Pendikom berbeda.\')\n    audit=files.frame(\'OUTPUT/SPLIT/PENDIKOM/70_30_groundtruth_feature_audit.csv\',True)\n    if audit.loc[audit.used.str.lower().eq(\'true\'),\'feature\'].tolist()!=s.PEND:raise RuntimeError(\'Feature mask Pendikom berubah.\')\n    tr=files.frame(\'OUTPUT/SPLIT/PENDIKOM/TRAIN/categorical.csv\',True);te=files.frame(\'OUTPUT/SPLIT/PENDIKOM/TEST/categorical.csv\',True)\n    rt=files.frame(\'OUTPUT/SPLIT/PENDIKOM/TRAIN/reference_ils.csv\',True);re=files.frame(\'OUTPUT/SPLIT/PENDIKOM/TEST/reference_ils.csv\',True)\n    if (tr.columns.tolist()!=[\'userid\',\'courseid\',\'split\']+s.PEND or te.columns.tolist()!=tr.columns.tolist()\n        or len(tr)!=322 or len(te)!=139 or not tr.userid.is_unique or not te.userid.is_unique or set(tr.userid)&set(te.userid)\n        or not tr.split.eq(\'train\').all() or not te.split.eq(\'test\').all()):raise RuntimeError(\'Skema/split Pendikom tidak sesuai.\')\n    if not tr[[\'userid\',\'courseid\']].equals(rt[[\'userid\',\'courseid\']]) or not te[[\'userid\',\'courseid\']].equals(re[[\'userid\',\'courseid\']]):raise RuntimeError(\'Identitas reference ILS tidak sejajar.\')\n    if not np.isin(tr[s.PEND].to_numpy(),[\'1\',\'2\',\'3\']).all() or not np.isin(te[s.PEND].to_numpy(),[\'1\',\'2\',\'3\']).all():raise RuntimeError(\'Kategori prediktor tidak valid.\')\n    original=files.frame(\'LIB/pendikom_ils_labels.csv\',True).set_index([\'userid\',\'courseid\']);cols=[\'ils_\'+k+\'_code\' for k in s.KEYS]+[\'ils_ls_class\']\n    if not original.index.is_unique:raise RuntimeError(\'Reference ILS sumber tidak unik.\')\n    for frame,label in [(rt,\'train\'),(re,\'test\')]:\n        match=original.loc[list(zip(frame.userid,frame.courseid))].reset_index()\n        if not match[cols].equals(frame[cols]) or not match.split.eq(label).all():raise RuntimeError(\'Reference ILS berbeda dari protokol independen.\')\n    train_truth=frame_codes(s,rt);truth=frame_codes(s,re)\n    ctx={\'scope\':\'PENDIKOM_TRAIN\',\'dataset\':\'PENDIKOM\',\'rep\':None,\'features\':s.PEND,\n        \'X\':tr[s.PEND].to_numpy(object),\'test_X\':te[s.PEND].to_numpy(object),\'ids\':tr.userid.tolist(),\'test_ids\':te.userid.tolist(),\n        \'indices\':np.arange(322,dtype=np.int64),\'test_frame\':te}\n    ctx[\'matrix_sha256\']=s.digest({\'features\':s.PEND,\'rows\':ctx[\'X\'].tolist(),\'ids\':ctx[\'ids\'],\'indices\':ctx[\'indices\'].tolist(),\n        \'test_rows\':ctx[\'test_X\'].tolist(),\'test_ids\':ctx[\'test_ids\'],\'classes\':None})\n    up=s.upstream_job(files,p25,ctx,\'CARE-KM\',0)\n    rules=files.frame(s.RULE_FILES[\'PENDIKOM\'],True)[[\'feature\',\'dimension\',\'low_supports\',\'high_supports\']]\n    if sorted(map(tuple,rules.to_numpy()))!=s.expected_rules(\'PENDIKOM\'):raise RuntimeError(\'Evidence utama Pendikom berbeda.\')\n    targets,test_targets,ww=primary26(s,files,p26,ctx,up,rules)\n    # Validate the complete stage-27 prediction/bootstrap checkpoint against its inputs.\n    checkpoint=files.path(U27+\'/KONTROL/evaluasi_ils.json\');raw=s.read_checkpoint(checkpoint)\n    if not raw.get(\'signature\'):raise RuntimeError(\'Catatan evaluasi tahap 27 tidak tersedia.\')\n    record=s.cache_read(files,checkpoint,raw[\'signature\'])\n    if record is None:raise RuntimeError(\'Prediksi/bobot bootstrap tahap 27 berubah. Pulihkan atau jalankan tahap 27.\')\n    predpath=U27+\'/ILS/01_prediksi_per_peserta.csv.gz\';weightpath=U27+\'/ILS/09_bootstrap_weights.npz\'\n    pred=files.frame(predpath,True);r27=[\'PRIMARY_MODE_FIRST\',\'MEMBER_ONLY\',\'DIRECTION_CONSISTENT_MEMBER_FIRST\']\n    ids=[m+\'|\'+r for m in s.METHODS for r in r27]+[\'DIRECT_BEHAVIORAL_RULES\'];arrays={};runs={}\n    if set(pred.pathway)!=set(ids):raise RuntimeError(\'Jalur evaluasi sumber 27 tidak lengkap.\')\n    for pathway in ids:\n        frame=pred.loc[pred.pathway.eq(pathway)].copy();frame[\'run\']=frame.run.astype(int)\n        nrun=15 if pathway.split(\'|\')[0] in s.STOCHASTIC else 1\n        if sorted(frame.run.unique())!=list(range(nrun)):raise RuntimeError(\'Jumlah run sumber tidak sesuai.\')\n        rows=[]\n        for run in range(nrun):\n            part=frame.loc[frame.run.eq(run)]\n            if part.identity.tolist()!=ctx[\'test_ids\']:raise RuntimeError(\'Urutan prediksi sumber tidak sejajar.\')\n            codes=[]\n            for di,(key,poles) in enumerate(zip(s.KEYS,s.POLES)):\n                expected=np.where(truth[:,di]==1,poles[0],poles[1])\n                if not np.array_equal(part[\'reference_\'+key].to_numpy(),expected):raise RuntimeError(\'Reference prediksi 27 tidak sama dengan ILS test.\')\n                values=part[\'predicted_\'+key]\n                if not values.isin([*poles,\'UNRESOLVED\']).all():raise RuntimeError(\'Arah prediksi sumber tidak dikenal.\')\n                codes.append(np.where(values.eq(poles[0]),1,np.where(values.eq(poles[1]),-1,0)))\n            rows.append(np.column_stack(codes))\n        arrays[pathway]=np.asarray(rows,dtype=np.int8);runs[pathway]=nrun\n    signature=s.digest({\'code\':p27[\'code\'],\'truth\':truth.tolist(),\'ids\':re.userid.tolist(),\n        \'predictions\':{p:s.sha(arrays[p].tobytes()) for p in ids},\'runs\':runs,\n        \'n_boot\':p27[\'configuration\'][\'bootstrap_repetitions\'],\'seed\':p27[\'configuration\'][\'bootstrap_seed\']})\n    if signature!=record[\'signature\']:raise RuntimeError(\'Signature prediksi/bootstrap 27 tidak sesuai.\')\n    care=arrays[\'CARE-KM|PRIMARY_MODE_FIRST\'][0];direct=arrays[\'DIRECT_BEHAVIORAL_RULES\'][0]\n    if not np.array_equal(care,s.numeric_profiles(test_targets)):raise RuntimeError(\'Profil CARE-KM sumber tidak konsisten.\')\n    with np.load(io.BytesIO(files.read(weightpath)),allow_pickle=False) as weights:\n        W=weights[\'weights\'].copy();valid=weights[\'valid_draws\'].copy();w_ids=weights[\'learner_ids\'].copy();seed=int(weights[\'seed\'])\n    if (W.ndim!=2 or W.shape[1]!=139 or len(W)<10000 or len(valid)!=len(W) or not np.issubdtype(W.dtype,np.integer)\n        or not (W.sum(axis=1)==139).all() or not np.array_equal(w_ids,re.userid.to_numpy(str))):\n        raise RuntimeError(\'Bobot bootstrap sumber tidak sesuai peserta test.\')\n    pos=W.astype(float)@(truth==1);expected_valid=((pos>0)&(pos<139)).all(axis=1)\n    if not np.array_equal(valid,expected_valid) or valid.sum()<10000:raise RuntimeError(\'Valid draws bootstrap tidak sesuai.\')\n    watched={p:files.hash(p) for p in set(required) if not p.endswith(\'.ipynb\') and not p.endswith(\'status_eksekusi.json\')}\n    watched.update(up[\'watched\']);watched.update(ww);watched.update(record[\'outputs\'])\n    watched[U27+\'/KONTROL/evaluasi_ils.json\']=files.hash(U27+\'/KONTROL/evaluasi_ils.json\')\n    return {\'X_train\':tr[s.PEND].astype(int),\'X_test\':te[s.PEND].astype(int),\'train_ids\':tr.userid.tolist(),\'test_ids\':te.userid.tolist(),\n        \'test_courseids\':te.courseid.tolist(),\'y_cds\':targets.ls_class.to_numpy(str),\'y_ils\':rt.ils_ls_class.to_numpy(str),\n        \'train_truth\':train_truth,\'truth\':truth,\'reference\':re,\'care\':care,\'direct\':direct,\n        \'weights\':W,\'valid\':valid,\'weight_path\':weightpath,\'weight_seed\':seed,\'watched\':watched,\'code_checks\':checks}\n\n\ndef parameter_value(obj):\n    if obj is None or isinstance(obj,(str,int,float,bool)):return obj\n    if isinstance(obj,np.generic):return obj.item()\n    if isinstance(obj,(list,tuple)):return [parameter_value(x) for x in obj]\n    if isinstance(obj,dict):return {str(k):parameter_value(v) for k,v in obj.items()}\n    if isinstance(obj,type):return obj.__module__+\'.\'+obj.__qualname__\n    if hasattr(obj,\'get_params\'):return {\'estimator\':type(obj).__module__+\'.\'+type(obj).__qualname__,\'parameters\':parameter_value(obj.get_params(deep=False))}\n    raise TypeError(\'Parameter model belum didukung: \'+repr(type(obj)))\n\n\ndef build_models(factory_source):\n    from sklearn.preprocessing import OneHotEncoder\n    from sklearn.pipeline import Pipeline\n    from sklearn.naive_bayes import CategoricalNB\n    from sklearn.tree import DecisionTreeClassifier\n    from sklearn.svm import SVC\n    scope={\'OneHotEncoder\':OneHotEncoder,\'Pipeline\':Pipeline,\'CategoricalNB\':CategoricalNB,\'DecisionTreeClassifier\':DecisionTreeClassifier,\'SVC\':SVC,\'SPLIT_SEED\':161600}\n    exec(compile(factory_source,\'frozen_factory22\',\'exec\'),scope)\n    return scope[\'build_external_classifiers\']()\n\n\ndef fit_one(s,files,out,control,group,family,X_train,y_train,train_ids,code,factory_source,force):\n    """Only training data/labels enter fitting and its content signature."""\n    import joblib\n    from threadpoolctl import threadpool_limits\n    model=build_models(factory_source)[family];params=parameter_value(model)\n    training={\'features\':X_train.columns.tolist(),\'X\':X_train.to_numpy().tolist(),\'ids\':train_ids,\'y\':y_train.tolist()}\n    data_sha=s.digest(training);signature=s.digest({\'code\':code,\'group\':group,\'family\':family,\'training_sha256\':data_sha,\n        \'parameters\':params,\'rng_policy\':\'temporary numpy seed 161600 during fit; restore caller state\'})\n    checkpoint=control/\'MODELS\'/(group+\'_\'+family+\'.json\');path=out/\'MODELS\'/(group+\'_\'+family+\'.joblib\')\n    record=s.cache_read(files,checkpoint,signature,force)\n    if record:\n        bundle=joblib.load(io.BytesIO(files.read(str(path.relative_to(files.root)).replace(\'\\\\\',\'/\'))))\n        if bundle[\'training_sha256\']!=data_sha or bundle[\'features\']!=X_train.columns.tolist() or bundle[\'parameters\']!=params:\n            raise RuntimeError(\'Metadata model tersimpan tidak sesuai train.\')\n        return bundle,record,\'CACHE\'\n    previous=np.random.get_state();start=time.perf_counter()\n    try:\n        np.random.seed(161600)\n        with threadpool_limits(limits=1):model.fit(X_train.to_numpy() if family==\'NB\' else X_train,y_train)\n    finally:np.random.set_state(previous)\n    elapsed=time.perf_counter()-start\n    if not set(model.classes_)<=set(s.LS_LOOKUP.values()):raise RuntimeError(\'Model menghasilkan ruang label yang tidak dikenal.\')\n    bundle={\'model\':model,\'group\':group,\'family\':family,\'features\':X_train.columns.tolist(),\'parameters\':params,\n        \'training_sha256\':data_sha,\'n_train\':len(y_train),\'target_classes\':sorted(set(y_train)),\'rng_seed\':161600,\n        \'ILS_train_learners\':322 if group==\'ILS\' else 0,\'code\':code}\n    buffer=io.BytesIO();joblib.dump(bundle,buffer,compress=3);s.atomic(path,buffer.getvalue())\n    record=s.cache_write(files,checkpoint,signature,[path],{\'group\':group,\'family\':family,\'training_sha256\':data_sha,\n        \'fit_seconds\':elapsed,\'parameters\':params,\'n_train\':len(y_train),\'n_classes\':len(set(y_train))})\n    return bundle,record,\'FIT\'\n\n\ndef predict_one(s,files,out,control,bundle,record,X_test,test_ids,courseids,code,force):\n    group,family=bundle[\'group\'],bundle[\'family\'];model_hash=next(iter(record[\'outputs\'].values()))\n    sig=s.digest({\'code\':code,\'model_sha256\':model_hash,\'features\':X_test.columns.tolist(),\'X_test\':X_test.to_numpy().tolist(),\'test_ids\':test_ids})\n    checkpoint=control/\'PREDICTIONS\'/(group+\'_\'+family+\'.json\');path=out/\'PREDICTIONS\'/(group+\'_\'+family+\'.csv\')\n    cached=s.cache_read(files,checkpoint,sig,force)\n    if cached:\n        frame=files.frame(str(path.relative_to(files.root)).replace(\'\\\\\',\'/\'),True);dispo=\'CACHE\'\n    else:\n        pred=bundle[\'model\'].predict(X_test.to_numpy() if family==\'NB\' else X_test)\n        to_poles={v:k for k,v in s.LS_LOOKUP.items()}\n        if not set(pred)<=set(to_poles):raise RuntimeError(\'Prediksi kelas di luar taxonomy FSLSM.\')\n        frame=pd.DataFrame({\'identity\':test_ids,\'courseid\':courseids,\'predicted_ls_class\':pred})\n        for di,key in enumerate(s.KEYS):frame[key]=[s.ABBR[to_poles[v][di]] for v in pred]\n        s.save_csv(path,frame);s.cache_write(files,checkpoint,sig,[path],{\'group\':group,\'family\':family});dispo=\'PREDICT\'\n    if frame.identity.tolist()!=test_ids:raise RuntimeError(\'Urutan prediksi model berbeda dari test.\')\n    return s.numeric_profiles(frame),dispo\n\n\ndef metadata(pathway):\n    if pathway.startswith(\'CDS_\'):return {\'pathway\':pathway,\'family\':pathway[4:],\'training_target\':\'cluster-derived Base LS\',\'ILS_train_learners\':0,\'n_model_training_targets\':322}\n    if pathway in [\'ILS_NB\',\'ILS_DT\',\'ILS_SVM\']:return {\'pathway\':pathway,\'family\':pathway[4:],\'training_target\':\'ILS Base LS\',\'ILS_train_learners\':322,\'n_model_training_targets\':322}\n    if pathway==\'ILS_TRAIN_MAJORITY\':return {\'pathway\':pathway,\'family\':\'TRAIN_MAJORITY\',\'training_target\':\'majority pole from ILS train\',\'ILS_train_learners\':322,\'n_model_training_targets\':322}\n    return {\'pathway\':pathway,\'family\':\'DIRECT\',\'training_target\':\'behavioral evidence / frozen profile\',\'ILS_train_learners\':0,\'n_model_training_targets\':0}\n\n\ndef evaluate(s,files,out,control,data,predictions,code,force):\n    truth=data[\'truth\'];n=len(truth);source_weights_hash=files.hash(data[\'weight_path\'])\n    sig=s.digest({\'code\':code,\'truth\':truth.tolist(),\'test_ids\':data[\'test_ids\'],\n        \'predictions\':{p:s.sha(np.asarray(predictions[p],np.int8).tobytes()) for p in PATHS},\'source_weights_sha256\':source_weights_hash})\n    checkpoint=control/\'evaluasi_ils.json\';folder=out/\'ILS\';record=s.cache_read(files,checkpoint,sig,force)\n    if record:\n        print(\'Evaluasi/CI: gunakan hasil yang isinya terverifikasi sama.\',flush=True)\n        return files.frame(str((folder/\'04_ringkasan_overall.csv\').relative_to(files.root)).replace(\'\\\\\',\'/\')),True\n    from threadpoolctl import threadpool_limits\n    W=data[\'weights\'][data[\'valid\']].astype(float);B=len(W);boot=np.empty((B,len(PATHS),5,3));points={};counts={}\n    print(\'Evaluasi:\',len(PATHS),\'jalur;\',B,\'bobot bootstrap berpasangan dari tahap 27.\',flush=True)\n    with threadpool_limits(limits=1):\n        for pi,pathway in enumerate(PATHS):\n            array=np.asarray(predictions[pathway],dtype=np.int8)[None,:,:]\n            mask=s.confusion_masks(truth,array)[:,0,:,:];cc=mask.sum(axis=0)\n            counts[pathway]=cc;points[pathway]=s.count_metrics(cc)\n            sampled=(W@mask.reshape(n,-1)).reshape(B,4,6);metrics=s.count_metrics(sampled)\n            boot[:,pi,:4]=metrics;boot[:,pi,4]=metrics.mean(axis=1)\n    per_dim=[];overall=[];cm=[];predrows=[];refcounts=[]\n    for di,(key,poles) in enumerate(zip(s.KEYS,s.POLES)):\n        for value,pole in [(1,poles[0]),(-1,poles[1])]:\n            refcounts.append({\'dimension\':key,\'reference_pole\':pole,\'n_train\':int((data[\'train_truth\'][:,di]==value).sum()),\n                \'n_test\':int((truth[:,di]==value).sum())})\n    for pi,pathway in enumerate(PATHS):\n        meta=metadata(pathway);arr=np.asarray(predictions[pathway]);point=points[pathway]\n        for li,identity in enumerate(data[\'test_ids\']):\n            row={**meta,\'identity\':identity,\'courseid\':data[\'test_courseids\'][li]};combo=[]\n            for di,(key,poles) in enumerate(zip(s.KEYS,s.POLES)):\n                value=arr[li,di];pole=poles[0] if value==1 else poles[1] if value==-1 else None\n                row[\'reference_\'+key]=poles[0] if truth[li,di]==1 else poles[1];row[\'predicted_\'+key]=pole or \'UNRESOLVED\';combo.append(pole)\n            row[\'predicted_ls_class\']=s.LS_LOOKUP[tuple(combo)] if all(combo) else \'AMBIGUOUS\'\n            row[\'reference_ls_class\']=data[\'reference\'].ils_ls_class.iloc[li];predrows.append(row)\n        for di,(key,poles) in enumerate(zip(s.KEYS,s.POLES)):\n            for ci,num in enumerate(counts[pathway][di]):cm.append({**meta,\'dimension\':key,\'reference_pole\':poles[ci//3],\n                \'predicted_pole\':(poles+(\'UNRESOLVED\',))[ci%3],\'n\':int(num)})\n        for di,key in enumerate(s.KEYS+[\'mean_four_dimensions\']):\n            values=point[di] if di<4 else point.mean(axis=0)\n            row={**meta,\'dimension\':key,\'n_test\':139,\'n_runs\':1,\'n_bootstrap\':B,\n                \'coverage\':float((arr[:,di]!=0).mean()) if di<4 else float((arr!=0).mean()),\n                \'complete_profile_coverage\':float((arr!=0).all(axis=1).mean()),\n                \'complete_profile_accuracy\':float(((arr==truth)&(arr!=0)).all(axis=1).mean())}\n            for ji,metric in enumerate(s.METRIC_NAMES):\n                row[metric]=values[ji];row[metric+\'_ci95_low\'],row[metric+\'_ci95_high\']=np.quantile(boot[:,pi,di,ji],[.025,.975])\n            (per_dim if di<4 else overall).append(row)\n    pairs=[];secondary=[]\n    for family in FAMILIES:\n        a=\'ILS_\'+family;ai=PATHS.index(a)\n        for b in [\'CDS_\'+family,\'CARE_DIRECT\',\'BEHAVIORAL_DIRECT\']:\n            bi=PATHS.index(b);primary=b.startswith(\'CDS_\')\n            for di,key in enumerate(s.KEYS+[\'mean_four_dimensions\']):\n                for ji,metric in enumerate(s.METRIC_NAMES):\n                    av=points[a][di,ji] if di<4 else points[a][:,ji].mean();bv=points[b][di,ji] if di<4 else points[b][:,ji].mean()\n                    delta=boot[:,ai,di,ji]-boot[:,bi,di,ji]\n                    row={\'family\':family,\'supervised_pathway\':a,\'comparison_pathway\':b,\'dimension\':key,\'metric\':metric,\n                        \'ILS_minus_comparison\':av-bv,\'paired_ci95_low\':np.quantile(delta,.025),\'paired_ci95_high\':np.quantile(delta,.975),\n                        \'positive_favors\':\'supervised ILS\',\'n_learners\':139,\'ILS_train_learners\':322,\'comparison_ILS_train_learners\':0,\n                        \'primary_family\':primary and di==4 and metric==\'balanced_accuracy\'}\n                    if row[\'primary_family\']:\n                        row[\'bonferroni_family95_low\'],row[\'bonferroni_family95_high\']=np.quantile(delta,[.05/6,1-.05/6])\n                        row[\'family_definition\']=\'three matched classifier label-source contrasts, overall mean BA\'\n                    (pairs if primary else secondary).append(row)\n    paths=[]\n    for name,frame in [(\'01_prediksi_semua_jalur.csv\',pd.DataFrame(predrows)),(\'02_metrik_per_dimensi.csv\',pd.DataFrame(per_dim)),\n        (\'04_ringkasan_overall.csv\',pd.DataFrame(overall)),(\'05_confusion_matrices.csv\',pd.DataFrame(cm)),(\'08_reference_counts.csv\',pd.DataFrame(refcounts)),\n        (\'PAIRWISE/01_ILS_vs_CDS.csv\',pd.DataFrame(pairs)),(\'PAIRWISE/02_ILS_vs_profil_langsung.csv\',pd.DataFrame(secondary))]:\n        path=folder/name;s.save_csv(path,frame);paths.append(path)\n    buf=io.BytesIO();np.savez_compressed(buf,metrics=boot,pathways=np.asarray(PATHS),dimensions=np.asarray(s.KEYS+[\'mean_four_dimensions\']),metric_names=np.asarray(s.METRIC_NAMES))\n    path=folder/\'06_bootstrap_paired_metrics.npz\';s.atomic(path,buf.getvalue());paths.append(path)\n    path=folder/\'07_protokol_evaluasi.json\';s.save_json(path,{\'primary_metric\':\'mean balanced accuracy over four ILS dimensions\',\n        \'primary_pairs\':[\'ILS_NB vs CDS_NB\',\'ILS_DT vs CDS_DT\',\'ILS_SVM vs CDS_SVM\'],\'n_learners\':139,\'n_bootstrap\':B,\n        \'weights_source\':data[\'weight_path\'],\'weights_sha256\':source_weights_hash,\'bootstrap_unit\':\'same learner weights for all pathways; four dimensions remain paired\',\n        \'individual_CIs\':\'descriptive paired percentile 95%\',\'family_intervals\':\'Bonferroni percentile family95, three overall mean-BA matched-model contrasts\',\n        \'secondary_pairs\':\'ILS classifier vs direct profiles, descriptive with different label budgets\',\n        \'macro_F1\':\'binary poles per dimension, then mean over four dimensions; not 14-class composite macro-F1\',\n        \'unresolved\':\'abstention included as error; no test learners dropped\',\n        \'supervision_budget\':\'ILS targets for 322 train learners only in ILS/majority group; CDS uses constructed targets\',\n        \'stratification\':\'original shared split used ILS; zero means no ILS target in model training, not zero access throughout study\',\n        \'scope\':\'conditional on fixed train/test split, features and model configurations; same holdout, not a new cohort\'})\n    paths.append(path);s.cache_write(files,checkpoint,sig,paths,{\'n_pathways\':9,\'n_learners\':139,\'n_bootstrap\':B})\n    return pd.DataFrame(overall),False\n\n\ndef figures(s,out,summary,pairwise):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    frame=summary.set_index(\'pathway\');fig,ax=plt.subplots(figsize=(8.5,4.8));y=np.arange(3)\n    for group,offset,color in [(\'CDS\',-.12,\'#555555\'),(\'ILS\',.12,\'#185785\')]:\n        values=frame.loc[[group+\'_\'+f for f in FAMILIES]];yy=y+offset\n        ax.hlines(yy,100*values.balanced_accuracy_ci95_low,100*values.balanced_accuracy_ci95_high,color=color)\n        ax.plot(100*values.balanced_accuracy,yy,\'o\',color=color,label=group+\' targets (\'+(\'0\' if group==\'CDS\' else \'322\')+\' ILS-labelled train learners)\')\n    ax.set_yticks(y,FAMILIES);ax.invert_yaxis();ax.set_xlim(0,100);ax.set_xlabel(\'Mean balanced accuracy (%)\')\n    ax.set_title(\'Same classifiers and features; different training supervision\');ax.legend(loc=\'lower left\',fontsize=10);fig.tight_layout()\n    folder=out/\'GAMBAR\';folder.mkdir(parents=True,exist_ok=True);paths=[]\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'01_perbandingan_sumber_supervisi.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig)\n    frame=pairwise.loc[pairwise.primary_family.eq(True)].set_index(\'family\').loc[FAMILIES]\n    fig,ax=plt.subplots(figsize=(8,4));ax.axvline(0,color=\'#888888\',linewidth=1)\n    ax.hlines(y,100*frame.bonferroni_family95_low,100*frame.bonferroni_family95_high,color=\'#333333\')\n    ax.plot(100*frame.ILS_minus_comparison,y,\'o\',color=\'#185785\');ax.set_yticks(y,FAMILIES);ax.invert_yaxis()\n    ax.set_xlabel(\'ILS minus CDS mean balanced accuracy (percentage points)\')\n    ax.set_title(\'Paired Bonferroni family 95% intervals: three label-source contrasts\');fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'02_selisih_berpasangan_supervisi.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig);return paths\n\n\ndef run_supervised(root,configuration,embedded_source,factory_source,s):\n    if set(configuration)-set(DEFAULT_CONFIG):raise ValueError(\'Konfigurasi tahap 28 tidak dikenal.\')\n    cfg=dict(DEFAULT_CONFIG);cfg.update(configuration)\n    if not isinstance(cfg[\'force\'],bool):raise ValueError(\'force harus True/False.\')\n    files=s.Files(Path(root).expanduser().resolve());data=prepare(s,files,factory_source)\n    own=s.notebook_hash(files,STAGE,\'supervised28_configuration\',\'KONFIGURASI_28\')\n    checks=data[\'code_checks\']+[(STAGE,\'supervised28_configuration\',\'KONFIGURASI_28\',own)]\n    import joblib\n    code={\'implementation_sha256\':s.sha(embedded_source.encode(\'utf-8\')),\'helper26_sha256\':s.sha(s.EMBEDDED_SOURCE.encode(\'utf-8\')),\n        \'factory22_sha256\':s.sha(factory_source.encode(\'utf-8\')),\'notebook_code\':own,\'runtime\':{**s.runtime(),\'joblib\':joblib.__version__}}\n    out=files.root/\'OUTPUT/TAMBAHAN\'/STAGE;control=out/\'KONTROL\';out.mkdir(parents=True,exist_ok=True);control.mkdir(parents=True,exist_ok=True)\n    s.save_json(out/\'PROTOKOL/01_konfigurasi.json\',{\'configuration\':cfg,\'code\':code,\'features\':s.PEND,\'train_rows\':322,\'test_rows\':139,\n        \'model_design\':\'three matched classifier families, paired CDS and ILS targets; same features, preprocessing and configuration\',\n        \'target_design\':\'composite Base LS then four-direction decomposition, same as manuscript Pendikom classifier pathway\',\n        \'fit_policy\':\'save six paired models once; cache by training contents and configuration; no clustering/profile refit\',\n        \'label_budget\':\'CDS 0 ILS training targets; ILS 322 learners/1288 directions; original shared split used ILS stratification\',\n        \'majority_policy\':\'per-dimension majority from ILS train; tie chooses first predefined pole\',\n        \'primary_inference\':\'three matched-family ILS-vs-CDS mean-BA contrasts; shared explicit weights from 27\',\n        \'upstream_weights\':data[\'weight_path\'],\'new_clustering_runs\':0})\n    class_rows=[]\n    for group,key in [(\'CDS\',\'y_cds\'),(\'ILS\',\'y_ils\')]:\n        for cls,n in pd.Series(data[key]).value_counts().sort_index().items():class_rows.append({\'supervision\':group,\'ls_class\':cls,\'n_train\':int(n)})\n    s.save_csv(out/\'PROTOKOL/02_target_counts.csv\',pd.DataFrame(class_rows))\n    predictions={\'CARE_DIRECT\':data[\'care\'],\'BEHAVIORAL_DIRECT\':data[\'direct\']};model_counts={\'FIT\':0,\'CACHE\':0};pred_counts={\'PREDICT\':0,\'CACHE\':0};models=[]\n    try:\n        for group,key in [(\'CDS\',\'y_cds\'),(\'ILS\',\'y_ils\')]:\n            for family in FAMILIES:\n                s.unchanged(files,data[\'watched\'],checks)\n                print(\'Model:\',group,family,\'| train=322, features=18\',flush=True)\n                bundle,record,dispo=fit_one(s,files,out,control,group,family,data[\'X_train\'],data[key],data[\'train_ids\'],code,factory_source,cfg[\'force\'])\n                model_counts[dispo]+=1\n                pred,dp=predict_one(s,files,out,control,bundle,record,data[\'X_test\'],data[\'test_ids\'],data[\'test_courseids\'],code,cfg[\'force\'])\n                predictions[group+\'_\'+family]=pred;pred_counts[dp]+=1\n                models.append({\'pathway\':group+\'_\'+family,\'family\':family,\'supervision\':group,\'ILS_train_learners\':322 if group==\'ILS\' else 0,\n                    \'n_train\':322,\'n_classes\':len(bundle[\'target_classes\']),\'model_file\':next(iter(record[\'outputs\'])),\'model_sha256\':next(iter(record[\'outputs\'].values())),\n                    \'training_sha256\':bundle[\'training_sha256\'],\'parameters\':json.dumps(bundle[\'parameters\'],sort_keys=True),\'fit_seconds\':record.get(\'fit_seconds\')})\n                s.save_json(control/\'status_eksekusi.json\',{\'status\':\'berjalan\',\'model_counts\':model_counts,\'prediction_counts\':pred_counts,\'last_model\':group+\'_\'+family})\n        if len(models)!=6:raise RuntimeError(\'Enam model pasangan belum lengkap.\')\n        majority=np.where((data[\'train_truth\']==1).sum(axis=0)>=(data[\'train_truth\']==-1).sum(axis=0),1,-1)\n        predictions[\'ILS_TRAIN_MAJORITY\']=np.repeat(majority[None,:],139,axis=0)\n        s.save_json(out/\'BASELINES/01_ils_train_majority.json\',{\'train_counts_first\':(data[\'train_truth\']==1).sum(axis=0),\n            \'train_counts_second\':(data[\'train_truth\']==-1).sum(axis=0),\'pole_codes\':majority,\'dimensions\':s.KEYS,\'n_train\':322,\'source\':\'ILS train only\'})\n        s.unchanged(files,data[\'watched\'],checks)\n        summary,cached=evaluate(s,files,out,control,data,predictions,code,cfg[\'force\'])\n        s.save_csv(out/\'MODELS/00_peta_model.csv\',pd.DataFrame(models))\n        table=summary.copy()\n        for metric in s.METRIC_NAMES:table[metric+\'_mean_ci95\']=table.apply(lambda r:f"{100*r[metric]:.2f} [{100*r[metric+\'_ci95_low\']:.2f}, {100*r[metric+\'_ci95_high\']:.2f}]",axis=1)\n        s.save_csv(out/\'TABEL/01_referensi_supervised_ILS.csv\',table[[\'pathway\',\'family\',\'training_target\',\'ILS_train_learners\',\'n_model_training_targets\']+\n            [m+\'_mean_ci95\' for m in s.METRIC_NAMES]+[\'coverage\',\'complete_profile_accuracy\']])\n        paired=pd.read_csv(out/\'ILS/PAIRWISE/01_ILS_vs_CDS.csv\',float_precision=\'round_trip\')\n        s.save_csv(out/\'TABEL/02_selisih_utama_mean_BA.csv\',paired.loc[paired.primary_family.eq(True)])\n        figsig=s.digest({\'code\':code,\'summary\':s.sha((out/\'ILS/04_ringkasan_overall.csv\').read_bytes()),\'paired\':s.sha((out/\'ILS/PAIRWISE/01_ILS_vs_CDS.csv\').read_bytes())})\n        if not s.cache_read(files,control/\'gambar.json\',figsig,cfg[\'force\']):s.cache_write(files,control/\'gambar.json\',figsig,figures(s,out,summary,paired),{})\n        s.unchanged(files,data[\'watched\'],checks);s.save_json(out/\'PROTOKOL/03_manifest_masukan.json\',data[\'watched\'])\n        status={\'status\':\'selesai\',\'model_counts\':model_counts,\'prediction_counts\':pred_counts,\'paired_models\':6,\'new_supervised_ILS_models\':3,\n            \'n_unique_pathways\':9,\'n_train\':322,\'n_test\':139,\'n_bootstrap\':int(data[\'valid\'].sum()),\'evaluation_cached\':cached,\n            \'new_clustering_runs\':0,\'full_protocol\':True,\'output\':str(out),\'finished_utc\':datetime.now(timezone.utc).isoformat()}\n        manifest={str(p.relative_to(files.root)).replace(\'\\\\\',\'/\'):s.sha(p.read_bytes()) for p in out.rglob(\'*\') if p.is_file() and control not in p.parents}\n        s.save_json(control/\'manifest_keluaran.json\',manifest);s.save_json(control/\'status_eksekusi.json\',status)\n        print(\'Tahap 28 selesai:\',model_counts,\'| prediksi:\',pred_counts,\'| evaluasi cache:\',cached,flush=True)\n        print(\'Hasil:\',out)\n        return {\'summary\':summary,\'table\':table,\'paired\':paired.loc[paired.primary_family.eq(True)],\'status\':status,\'output\':str(out)}\n    except BaseException as error:\n        s.save_json(control/\'status_eksekusi.json\',{\'status\':\'terhenti\',\'model_counts\':model_counts,\'prediction_counts\':pred_counts,\n            \'error\':type(error).__name__+\': \'+str(error),\'resume\':\'jalankan notebook 28 kembali; isi cache model/prediksi/evaluasi diperiksa\'})\n        raise\n'

_supervised28 = types.ModuleType('_supervised28')
exec(compile(SUPERVISED28_SOURCE, '_supervised28', 'exec'), _supervised28.__dict__)


In [5]:
HASIL_28 = _supervised28.run_supervised(ROOT_28, KONFIGURASI_28,
                                      SUPERVISED28_SOURCE, FACTORY22_SOURCE, _s26)
display(HASIL_28['table'][['pathway', 'training_target', 'ILS_train_learners',
                          'accuracy_mean_ci95', 'balanced_accuracy_mean_ci95',
                          'macro_f1_mean_ci95']])
display(HASIL_28['paired'])
print(HASIL_28['status'])
print('Folder hasil:', HASIL_28['output'])


Model: CDS NB | train=322, features=18
Model: CDS DT | train=322, features=18
Model: CDS SVM | train=322, features=18
Model: ILS NB | train=322, features=18
Model: ILS DT | train=322, features=18
Model: ILS SVM | train=322, features=18
Evaluasi: 9 jalur; 10000 bobot bootstrap berpasangan dari tahap 27.
Tahap 28 selesai: {'FIT': 6, 'CACHE': 0} | prediksi: {'PREDICT': 6, 'CACHE': 0} | evaluasi cache: False
Hasil: /content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/28_Referensi_Supervised_ILS


,pathway,training_target,ILS_train_learners,accuracy_mean_ci95,balanced_accuracy_mean_ci95,macro_f1_mean_ci95
0,CARE_DIRECT,behavioral evidence / frozen profile,0,"89.75 [87.05, 92.27]","81.29 [76.52, 85.76]","82.98 [77.61, 87.28]"
1,BEHAVIORAL_DIRECT,behavioral evidence / frozen profile,0,"73.56 [69.78, 77.34]","71.76 [66.81, 76.55]","73.88 [68.73, 78.30]"
2,CDS_NB,cluster-derived Base LS,0,"90.83 [88.49, 92.99]","81.18 [76.75, 85.37]","83.99 [79.17, 87.76]"
3,CDS_DT,cluster-derived Base LS,0,"88.13 [85.61, 90.65]","78.25 [74.68, 81.78]","78.91 [74.61, 82.75]"
4,CDS_SVM,cluster-derived Base LS,0,"89.75 [87.05, 92.27]","80.24 [76.26, 84.12]","81.57 [76.54, 85.81]"
5,ILS_NB,ILS Base LS,322,"95.14 [93.17, 96.94]","90.18 [86.13, 93.81]","92.15 [88.43, 95.00]"
6,ILS_DT,ILS Base LS,322,"93.53 [91.73, 95.32]","89.97 [86.49, 93.22]","90.33 [87.04, 93.02]"
7,ILS_SVM,ILS Base LS,322,"93.35 [91.19, 95.32]","90.78 [87.01, 94.08]","90.77 [87.39, 93.54]"
8,ILS_TRAIN_MAJORITY,majority pole from ILS train,322,"75.72 [71.58, 79.68]","50.00 [50.00, 50.00]","42.98 [41.57, 44.24]"


,family,supervised_pathway,comparison_pathway,dimension,metric,ILS_minus_comparison,paired_ci95_low,paired_ci95_high,positive_favors,n_learners,ILS_train_learners,comparison_ILS_train_learners,primary_family,bonferroni_family95_low,bonferroni_family95_high,family_definition
13,NB,ILS_NB,CDS_NB,mean_four_dimensions,balanced_accuracy,0.090013,0.055539,0.127225,supervised ILS,139,322,0,True,0.048864,0.136239,three matched classifier label-source contrast...
28,DT,ILS_DT,CDS_DT,mean_four_dimensions,balanced_accuracy,0.117210,0.076617,0.158392,supervised ILS,139,322,0,True,0.067652,0.166479,three matched classifier label-source contrast...
43,SVM,ILS_SVM,CDS_SVM,mean_four_dimensions,balanced_accuracy,0.105422,0.062534,0.148140,supervised ILS,139,322,0,True,0.053716,0.156840,three matched classifier label-source contrast...


{'status': 'selesai', 'model_counts': {'FIT': 6, 'CACHE': 0}, 'prediction_counts': {'PREDICT': 6, 'CACHE': 0}, 'paired_models': 6, 'new_supervised_ILS_models': 3, 'n_unique_pathways': 9, 'n_train': 322, 'n_test': 139, 'n_bootstrap': 10000, 'evaluation_cached': False, 'new_clustering_runs': 0, 'full_protocol': True, 'output': '/content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/28_Referensi_Supervised_ILS', 'finished_utc': '2026-11-06T13:45:43.819465+00:00'}
Folder hasil: /content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/28_Referensi_Supervised_ILS


## Membaca keluaran

- `TABEL/01_referensi_supervised_ILS.csv`: sembilan jalur, asal target,
  jumlah peserta berlabel ILS untuk model, mean metrik dan CI 95%.
- `TABEL/02_selisih_utama_mean_BA.csv`: tiga kontras sumber supervisi utama.
- `ILS/`: prediksi per peserta, metrik per dimensi, confusion matrices,
  counts train/test, ringkasan, bank bootstrap, dan paired differences.
- `MODELS/`: enam estimator/pipeline `.joblib` serta peta fitur, parameter,
  sumber label, hash train, dan asal waktu fitting. Encoder SVM ikut disimpan.
- `PREDICTIONS/`: prediksi enam model; label test tidak diperlukan saat predict.
- `BASELINES/`: counts dan keputusan mayoritas dari ILS train.
- `GAMBAR/`: perbandingan model dan selisih berpasangan, PNG/PDF.
- `PROTOKOL/`, `KONTROL/`: label counts, konfigurasi, manifest, serta cache.

Model dapat dipakai kembali untuk eksperimen inference berikutnya dengan
urutan fitur dan versi lingkungan yang sesuai. Runtime komparatif terkontrol
tetap diukur pada eksperimen runtime; waktu model cached tidak diperlakukan
sebagai waktu fitting baru.

Notebook ini tidak mengulang delapan initializer: classifier supervised
tidak memiliki initializer. Main LMS dan benchmark kategorikal tidak
memiliki reference ILS yang digunakan untuk eksperimen ini.
